In [ ]:
%%sql -r dataframe_1

SELECT * FROM (
WITH event_info AS (
    SELECT
          de.EVENT_ID
        , de.EVENT_START_DT
        , de.EVENT_NAME
    FROM DWH.D_EVENT de
    WHERE de.EVENT_START_DT BETWEEN DATEADD(HOUR, -2 , CURRENT_TIMESTAMP()) AND DATEADD(HOUR, 5, CURRENT_TIMESTAMP())
),
fts AS (
    SELECT
          fts.EVENT_ID
        , fts.tournament_id
        , fts.OUTCOME_ID
        , fts.MARKET_FEED_CODE
        , fts.SELECTION_ODD_TYPE
        , fts.SELECTION_COEFFICIENT
        , fts.Accepted_DT
        ,dp.BUSINESS_DOMAIN_CODE
        , SUM(
            ((fts.STAKE_NET_AMT * fer1.MIDDLE_RATE)
             * CASE WHEN (fts.INITIAL_TICKET_SELECTION_COEFFICIENT_SUM = fts.SELECTION_COUNT)
                    THEN 1
                    ELSE (fts.INITIAL_SELECTION_COEFFICIENT) END)
            / CASE WHEN (fts.INITIAL_TICKET_SELECTION_COEFFICIENT_SUM = fts.SELECTION_COUNT)
                   THEN fts.SELECTION_COUNT
                   ELSE (fts.INITIAL_TICKET_SELECTION_COEFFICIENT_SUM - fts.SELECTION_COUNT) END
          ) AS stake_eur
        , SUM(
            CASE WHEN (fts.ORIGINAL_RESOLVED_DT IS NOT NULL)
                 THEN (((fts.STAKE_NET_AMT * fer1.MIDDLE_RATE
                          - (fts.PAYOUT_TOTAL_AMT  * fer2.MIDDLE_RATE
                             - fts.PAYOUT_TAX_GENEROSITY_AMT * fer2.MIDDLE_RATE))
                          * CASE WHEN (fts.INITIAL_TICKET_SELECTION_COEFFICIENT_SUM = fts.SELECTION_COUNT)
                                 THEN 1
                                 ELSE (fts.INITIAL_SELECTION_COEFFICIENT ) END)
                         / CASE WHEN (fts.INITIAL_TICKET_SELECTION_COEFFICIENT_SUM = fts.SELECTION_COUNT)
                                THEN fts.SELECTION_COUNT
                                ELSE (fts.INITIAL_TICKET_SELECTION_COEFFICIENT_SUM - fts.SELECTION_COUNT) END)
                 ELSE 0 END
          ) AS ggr_eur
        , COUNT(DISTINCT fts.player_id) AS n_players
        , SUM(
            (CASE WHEN (fts.INITIAL_TICKET_SELECTION_COEFFICIENT_SUM = fts.SELECTION_COUNT)
                  THEN 1
                  ELSE (fts.INITIAL_SELECTION_COEFFICIENT) END)
            / CASE WHEN (fts.INITIAL_TICKET_SELECTION_COEFFICIENT_SUM = fts.SELECTION_COUNT)
                   THEN fts.SELECTION_COUNT
                   ELSE (fts.INITIAL_TICKET_SELECTION_COEFFICIENT_SUM - fts.SELECTION_COUNT) END
          ) AS ggr_tickets
     FROM DWH.F_TICKET_SELECTION fts
    INNER JOIN (
        SELECT PLAYER_ID,  BUSINESS_DOMAIN_CODE FROM DWH.D_PLAYER
        WHERE NVL(IS_TEST_ACCOUNT, 0) = 0
    ) dp ON fts.PLAYER_ID = dp.PLAYER_ID
    INNER JOIN event_info ei ON fts.EVENT_ID = ei.EVENT_ID
    LEFT JOIN DWH.F_EXCHANGE_RATE fer1
        ON  fer1.FROM_CURRENCY_ID = fts.CURRENCY_ID
        AND fer1.RATE_DATE        = fts.ACCEPTED_DT::DATE
        AND fer1.TO_CURRENCY_CODE = 'EUR'
    LEFT JOIN DWH.F_EXCHANGE_RATE fer2
        ON  fer2.FROM_CURRENCY_ID = fts.CURRENCY_ID
        AND fer2.RATE_DATE        = fts.ORIGINAL_RESOLVED_DT::DATE
        AND fer2.TO_CURRENCY_CODE = 'EUR'
    WHERE fts.IS_COUNT_RELEVANT    = 1
      AND fts.SELECTION_BET_TYPE   = 'Regular'
      AND fts.SELECTION_COUNT      = 1
      AND fts.ORIGINAL_RESOLVED_DT IS NULL
      AND fts.ACCEPTED_DT          BETWEEN DATEADD(HOUR, -3, CURRENT_TIMESTAMP()) AND  CURRENT_TIMESTAMP()
      AND fts.TICKET_STATUS        NOT IN ('WIN', 'LOST')
    GROUP BY
          fts.EVENT_ID
        , fts.tournament_id
        , fts.OUTCOME_ID
        , fts.MARKET_FEED_CODE
        , fts.SELECTION_ODD_TYPE
        , fts.SELECTION_COEFFICIENT
        , fts.Accepted_DT
        ,dp.BUSINESS_DOMAIN_CODE
)
,base AS (
    SELECT
          fts.EVENT_ID                                          AS EVENT_ID
        , de.EVENT_NAME
        , de.EVENT_START_DT AS REAL_START_TIME
        , CASE
              WHEN dt.TOURNAMENT_NAME ILIKE '%World Cup 2026%'
              THEN 'World Cup 2026'
              ELSE dt.TOURNAMENT_NAME
          END                                                   AS TOURNAMENT_NAME
        , fts.BUSINESS_DOMAIN_CODE                               AS BUSINESS_DOMAIN
        , do.MARKET_NAME
        , ds.SPORT_NAME
        , TRIM(SPLIT_PART(de.EVENT_NAME, '·', 1))               AS team1
        , TRIM(SPLIT_PART(de.EVENT_NAME, '·', 2))               AS team2
        , do.OUTCOME_NAME                                       AS outcome
        , fts.SELECTION_COEFFICIENT                             AS odds
        , fts.stake_eur
        , fts.ggr_eur
        , fts.ggr_tickets
        , fts.n_players
        , fts.stake_eur   * (fts.SELECTION_COEFFICIENT )     AS ggr_stake_row
        , fts.n_players   * (fts.SELECTION_COEFFICIENT )     AS ggr_player_row
        , DATEDIFF(MINUTE, de.EVENT_START_DT, fts.Accepted_DT)   AS minutes_after_start
        , LEAST(FLOOR(DATEDIFF(MINUTE, de.EVENT_START_DT, fts.Accepted_DT) / 10.0) * 10, 100)
                                                                  AS band_start_min
        , CONCAT(
              CAST(LEAST(FLOOR(DATEDIFF(MINUTE, de.EVENT_START_DT, fts.Accepted_DT) / 10.0) * 10, 100) AS VARCHAR)
            , ' to '
            , CAST(LEAST(FLOOR(DATEDIFF(MINUTE, de.EVENT_START_DT, fts.Accepted_DT) / 10.0) * 10, 100) + 10 AS VARCHAR)
            , ' mins'
          )                                                     AS acceptance_time_band
    FROM fts
    INNER JOIN event_info        de  ON de.EVENT_ID        = fts.EVENT_ID
    INNER JOIN DWH.D_TOURNAMENT  dt  ON fts.tournament_id  = dt.TOURNAMENT_ID
    INNER JOIN DWH.D_CATEGORY    dc  ON dc.CATEGORY_ID     = dt.CATEGORY_ID
    INNER JOIN DWH.D_SPORT       ds  ON ds.SPORT_ID        = dc.SPORT_ID
    LEFT  JOIN DWH.D_OUTCOME     do  ON do.OUTCOME_ID      = fts.OUTCOME_ID
    WHERE ds.SPORT_NAME IN ('Soccer', 'Tennis', 'Basketball', 'Volleyball')
        AND do.MARKET_NAME IN (
              'Total points (incl. OT)',
              'Match Betting',
              'Match Winner'
          )
        
        AND DATEDIFF(MINUTE, de.EVENT_START_DT, fts.Accepted_DT) BETWEEN -60 AND 240
)
SELECT
      TO_CHAR(b.REAL_START_TIME + interval '1 HOUR', 'YYYY-MM-DD HH24:MI')         AS REAL_START_TIME 
    , b.EVENT_NAME
    , b.TOURNAMENT_NAME
    , b.BUSINESS_DOMAIN
    , b.MARKET_NAME
    , b.SPORT_NAME
    , b.team1                                                            AS TEAM1
    , b.team2                                                            AS TEAM2
    , b.outcome                                                          AS OUTCOME
    , b.acceptance_time_band                                             AS ACCEPTANCE_TIME_BAND
    , AVG(b.odds)                                                            as odds
    , SUM(b.stake_eur)                                                   AS STAKE_EUR
    ,  SUM(b.ggr_stake_row)                                               AS GGR_STAKE
    , SUM(b.n_players)                                                   AS N_PLAYERS
    ,  SUM(b.ggr_player_row)                                              AS GGR_PLAYER
    , SUM(SUM(b.stake_eur))   OVER (PARTITION BY b.EVENT_ID)             AS STAKE_EUR_MATCH_TOTAL
    , SUM(SUM(b.n_players))   OVER (PARTITION BY b.EVENT_ID)             AS N_PLAYERS_MATCH_TOTAL
FROM base b
WHERE
    b.SPORT_NAME IN ('Soccer', 'Tennis', 'Basketball')
   --and b.BUSINESS_DOMAIN not in ('SB_PL')
--and b.BUSINESS_DOMAIN not in ('SB_BE','NG_BE')
    --and b.BUSINESS_DOMAIN not in ('SB_RO')
   -- and b.TOURNAMENT_NAME not in ('Club Friendly Games')
GROUP BY
      b.EVENT_ID
    , b.REAL_START_TIME  
    , b.EVENT_NAME
    , b.TOURNAMENT_NAME
    , b.BUSINESS_DOMAIN
    , b.MARKET_NAME
    , b.SPORT_NAME
    , b.team1
    , b.team2
    , b.outcome
    , b.acceptance_time_band
ORDER BY
      b.REAL_START_TIME
    , MIN(b.band_start_min)
    , STAKE_EUR DESC
)
 UNION ALL 

 SELECT * FROM (
WITH max_accepted AS (
    SELECT MAX(fts.ACCEPTED_DT) AS mx
    FROM DWH.F_TICKET_SELECTION fts
    WHERE fts.IS_COUNT_RELEVANT = 1
      AND fts.ORIGINAL_RESOLVED_DT IS NULL
      AND fts.ACCEPTED_DT BETWEEN DATEADD(HOUR, -1, CURRENT_TIMESTAMP()) AND CURRENT_TIMESTAMP()
)

, tickets_filtered AS (
    SELECT A.RECORD_CONTENT, A.RECORD_METADATA
    FROM PROD.RAW_SPORTSBOOK.TICKETS_V2 A,
         LATERAL FLATTEN(input => A.RECORD_CONTENT:selections) AS s2
    WHERE TO_TIMESTAMP_LTZ((A.RECORD_METADATA:CreateTime::INT) / 1000)
              >= (SELECT mx FROM max_accepted)
      AND A.RECORD_CONTENT:selection_count_type::STRING <> 'MULTIPLE'
      AND RECORD_METADATA:headers:"betting.message_type":value = 'TICKET_APPROVED'
      AND TO_TIMESTAMP_LTZ(
            (A.RECORD_CONTENT:events::OBJECT[(s2.value:event_id)]:real_start_time::INT) / 1000
          ) BETWEEN CURRENT_TIMESTAMP - INTERVAL '2 HOURS' AND CURRENT_TIMESTAMP + INTERVAL '1 HOURS'
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY LOWER(A.RECORD_CONTENT:id::VARCHAR)
        ORDER BY s2.index
    ) = 1
)

, flattened AS (
    SELECT
        LOWER(A.RECORD_CONTENT:id::VARCHAR)                                   AS TICKET_ID,
        A.RECORD_CONTENT:"status"::VARCHAR                                    AS TICKET_STATUS,
        TO_TIMESTAMP_LTZ((A.RECORD_METADATA:CreateTime::INT) / 1000)          AS CREATE_TIME,
        A.RECORD_CONTENT:target::VARCHAR                                      AS BUSINESS_DOMAIN,
        A.RECORD_CONTENT:config.currency::VARCHAR                             AS CONFIG_CURRENCY,
        LOWER(A.RECORD_CONTENT:player_id::VARCHAR)                            AS PLAYER_ID,
        A.RECORD_CONTENT:selection_count_type::VARCHAR                        AS SELECTION_COUNT_TYPE,
        LOWER(event.value:id::VARCHAR)                                        AS EVENT_ID,
        event.value:name::VARCHAR                                             AS EVENT_NAME,
        event.value:sport.name::VARCHAR                                       AS SPORT_NAME,
        event.value:category.name::VARCHAR                                    AS CATEGORY_NAME,
        event.value:tournament.name::VARCHAR                                  AS TOURNAMENT_NAME,
        TO_TIMESTAMP_LTZ(event.value:real_start_time::BIGINT / 1000)          AS REAL_START_TIME,
        f.value:key::VARCHAR                                                  AS SPECIFIER_KEY,
        f.value:value::VARCHAR                                                AS SPECIFIER_VALUE,
        A.RECORD_CONTENT:payment.stake.total::FLOAT                           AS PAYMENT_STAKE_TOTAL,
        A.RECORD_CONTENT:initial_potential_payout.coefficient::FLOAT          AS INITIALPOTENTIALPAYOUT_COEFFICIENT,
        A.RECORD_CONTENT:initial_potential_payout.potential_winning.amount::FLOAT
                                                                               AS INITIALPOTENTIALPAYOUT_POTENTIALWINNING_AMOUNT,
        A.RECORD_CONTENT:"system"."type"::VARCHAR                             AS SYSTEM_TYPE,
        A.RECORD_CONTENT:"system".combination_count::INT                      AS SYSTEM_COMBINATIONCOUNT,
        A.RECORD_CONTENT:event_type::VARCHAR                                  AS EVENTS_TYPE,
        LOWER(S.value:id::VARCHAR)                                            AS SELECTIONS_ID,
        S.value:"status"::VARCHAR                                            AS SELECTIONS_STATUS,
        S.value:odd.coefficient::FLOAT                                       AS SELECTIONS_ODD_COEFFICIENT,
        S.value:odd."status"::VARCHAR                                       AS SELECTIONS_ODD_STATUS,
        SPLIT_PART(S.value:market_line.legacy_name, '|', 1)::VARCHAR         AS SELECTIONS_MARKETLINE_LEGACY_NAME,
        SPLIT_PART(S.value:odd.legacy_name,        '|', 1)::VARCHAR          AS SELECTIONS_ODD_LEGACY_NAME,
        S.value:market_line."type"::VARCHAR                                  AS SELECTIONS_MARKETLINE_TYPE
    FROM tickets_filtered A,
         LATERAL FLATTEN(input => A.RECORD_CONTENT:selections, OUTER => TRUE) AS S,
         LATERAL FLATTEN(input => S.value:market_line.specifiers, OUTER => TRUE) AS f,
         LATERAL FLATTEN(input => A.RECORD_CONTENT:events, OUTER => TRUE) AS event
    WHERE S.value:odd."status"::VARCHAR = 'ACTIVE'
      AND event.value:id::VARCHAR = S.value:event_id::VARCHAR
)

, base AS (
    SELECT
        aa.* EXCLUDE (EVENT_NAME, SPORT_NAME, TOURNAMENT_NAME),
        de.EVENT_NAME,
        ds.SPORT_NAME,
        dt.TOURNAMENT_NAME,
        (aa.PAYMENT_STAKE_TOTAL * fx.MIDDLE_RATE)::FLOAT AS STAKE_EUR
    FROM flattened aa
    JOIN PROD.DWH.D_EVENT de
        ON de.EVENT_ID = aa.EVENT_ID
    JOIN PROD.DWH.D_TOURNAMENT dt
        ON dt.TOURNAMENT_ID = de.TOURNAMENT_ID
    JOIN PROD.DWH.D_CATEGORY dc
        ON dc.CATEGORY_ID = dt.CATEGORY_ID
    JOIN PROD.DWH.D_SPORT ds
        ON ds.SPORT_ID = dc.SPORT_ID
    LEFT JOIN DWH.F_EXCHANGE_RATE fx
        ON fx.FROM_CURRENCY_CODE = aa.CONFIG_CURRENCY
        AND fx.TO_CURRENCY_CODE  = 'EUR'
        AND fx.RATE_DATE = (
              SELECT MAX(fx2.RATE_DATE)
              FROM DWH.F_EXCHANGE_RATE fx2
              WHERE fx2.FROM_CURRENCY_CODE = aa.CONFIG_CURRENCY
                AND fx2.TO_CURRENCY_CODE   = 'EUR'
                AND fx2.RATE_DATE <= TO_DATE(aa.REAL_START_TIME)
        )
    WHERE de.EVENT_START_DT > CURRENT_TIMESTAMP - INTERVAL '3 HOUR'
      AND aa.SELECTIONS_MARKETLINE_LEGACY_NAME IN (
              'Money line (incl. OT)', 'Match Betting', 'Match Winner'
          )
      AND ds.SPORT_NAME IN ('Soccer', 'Basketball', 'Tennis', 'Volleyball')
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY aa.TICKET_ID, aa.SELECTIONS_ID
        ORDER BY de.EVENT_ID
    ) = 1
)

, base_windowed AS (
    SELECT
        base.*,
        LAST_VALUE(base.INITIALPOTENTIALPAYOUT_COEFFICIENT IGNORE NULLS) OVER (
            PARTITION BY base.EVENT_ID,
                         base.SELECTIONS_ODD_LEGACY_NAME,
                         base.SELECTIONS_MARKETLINE_LEGACY_NAME,
                         base.SELECTIONS_MARKETLINE_TYPE
            ORDER BY base.REAL_START_TIME
        ) AS LAST_INITIALPOTENTIALPAYOUT_COEFFICIENT,
        DATEDIFF('second', base.REAL_START_TIME, base.CREATE_TIME) / 60.0 AS MINS_FROM_START
    FROM base
    WHERE base.INITIALPOTENTIALPAYOUT_COEFFICIENT IS NOT NULL
      AND base.CREATE_TIME IS NOT NULL
      AND base.INITIALPOTENTIALPAYOUT_COEFFICIENT > 1.0
      AND base.STAKE_EUR > 0
      AND DATEDIFF('second', base.REAL_START_TIME, base.CREATE_TIME) / 60.0 BETWEEN -60 AND 240
)

, band_base AS (
    SELECT
        bw.EVENT_ID,
        bw.EVENT_NAME,
        bw.REAL_START_TIME,
        bw.TOURNAMENT_NAME,
        bw.BUSINESS_DOMAIN,
        bw.SELECTIONS_MARKETLINE_LEGACY_NAME              AS MARKET_NAME,
        bw.SPORT_NAME,
        TRIM(SPLIT_PART(bw.EVENT_NAME, '·', 1))           AS team1,
        TRIM(SPLIT_PART(bw.EVENT_NAME, '·', 2))           AS team2,
        bw.SELECTIONS_ODD_LEGACY_NAME                     AS outcome,
        bw.SELECTIONS_ODD_COEFFICIENT                     AS odds,
        bw.STAKE_EUR                                      AS stake_eur,
        bw.STAKE_EUR * bw.SELECTIONS_ODD_COEFFICIENT      AS ggr_stake_row,
        bw.TICKET_ID                                      AS PLayer_ID,
        LEAST(FLOOR(bw.MINS_FROM_START / 10.0) * 10, 100)  AS band_start_min,
        CONCAT(
            CAST(LEAST(FLOOR(bw.MINS_FROM_START / 10.0) * 10, 100) AS VARCHAR),
            ' to ',
            CAST(LEAST(FLOOR(bw.MINS_FROM_START / 10.0) * 10, 100) + 10 AS VARCHAR),
            ' mins'
        ) AS acceptance_time_band
    FROM base_windowed bw
)

, agg AS (
    SELECT
        EVENT_ID, EVENT_NAME, REAL_START_TIME, TOURNAMENT_NAME, BUSINESS_DOMAIN, MARKET_NAME, SPORT_NAME,
        team1, team2, outcome, acceptance_time_band, band_start_min,
        AVG(odds)                                  AS odds,
        SUM(stake_eur)                              AS stake_eur,
        SUM(ggr_stake_row)                          AS ggr_stake_row,
        COUNT(DISTINCT PLayer_ID)                   AS n_players,
        COUNT(DISTINCT PLayer_ID) * AVG(odds)       AS ggr_player_row
    FROM band_base
    GROUP BY EVENT_ID, EVENT_NAME, REAL_START_TIME, TOURNAMENT_NAME, BUSINESS_DOMAIN, MARKET_NAME, SPORT_NAME,
             team1, team2, outcome, acceptance_time_band, band_start_min
)

SELECT
    TO_CHAR(a.REAL_START_TIME + INTERVAL '1 HOUR', 'YYYY-MM-DD HH24:MI') AS REAL_START_TIME,
    a.EVENT_NAME,
    a.TOURNAMENT_NAME,
    a.BUSINESS_DOMAIN,
    a.MARKET_NAME,
    a.SPORT_NAME,
    a.team1                                                             AS TEAM1,
    a.team2                                                             AS TEAM2,
    a.outcome                                                           AS OUTCOME,
    a.acceptance_time_band                                              AS ACCEPTANCE_TIME_BAND,
    a.odds                                                              AS odds,
    a.stake_eur                                                         AS STAKE_EUR,
    a.ggr_stake_row                                                     AS GGR_STAKE,
    a.n_players                                                         AS N_PLAYERS,
    a.ggr_player_row                                                    AS GGR_PLAYER,
    SUM(a.stake_eur) OVER (PARTITION BY a.EVENT_ID)                     AS STAKE_EUR_MATCH_TOTAL,
    SUM(a.n_players) OVER (PARTITION BY a.EVENT_ID)                     AS N_PLAYERS_MATCH_TOTAL
FROM agg a
WHERE a.SPORT_NAME IN ('Soccer', 'Tennis', 'Basketball')
--ORDER BY REAL_START_TIME, a.band_start_min, STAKE_EUR DESC
)
;

In [ ]:
import re
import unicodedata
import difflib
import pandas as pd
import numpy as np
# ─── STEP 1: Assign 10-minute bands (22 bands: -60 to +150) ─────────────────
def assign_band(mins):
    if mins < -50:   return 1,  '−60 to −51 min'
    elif mins < -40: return 2,  '−50 to −41 min'
    elif mins < -30: return 3,  '−40 to −31 min'
    elif mins < -20: return 4,  '−30 to −21 min'
    elif mins < -10: return 5,  '−20 to −11 min'
    elif mins < 0:   return 6,  '−10 to  −1 min'
    elif mins < 10:  return 7,  '0 to  +9 min'
    elif mins < 20:  return 8,  '+10 to +19 min'
    elif mins < 30:  return 9,  '+20 to +29 min'
    elif mins < 40:  return 10, '+30 to +39 min'
    elif mins < 50:  return 11, '+40 to +49 min'
    elif mins < 60:  return 12, '+50 to +59 min'
    elif mins < 70:  return 13, '+60 to +69 min'
    elif mins < 80:  return 14, '+70 to +79 min'
    elif mins < 90:  return 15, '+80 to +89 min'
    elif mins < 100: return 16, '+90 to +99 min'
    elif mins < 110: return 17, '+100 to +109 min'
    elif mins < 120: return 18, '+110 to +119 min'
    elif mins < 130: return 19, '+120 to +129 min'
    elif mins < 140: return 20, '+130 to +139 min'
    elif mins < 150: return 21, '+140 to +149 min'
    else:            return 22, '+150 min or more'
def band_order_from_label(band_text: str) -> int:
    if pd.isna(band_text):
        return np.nan
    m = re.search(r"(-?\d+)\s*to", str(band_text))
    if m:
        start_min = int(m.group(1))
        return int((start_min + 60) // 10 + 1)
    return np.nan
df = dataframe_1.copy() if isinstance(dataframe_1, pd.DataFrame) else dataframe_1.to_pandas()
# Normalise column names from SQL (Snowflake returns uppercase)
df.columns = [c.upper() for c in df.columns]
df["BAND_ORDER"] = df["ACCEPTANCE_TIME_BAND"].apply(band_order_from_label)
df[['BAND_ORDER', 'BAND_LABEL']] = df['BAND_ORDER'].apply(
    lambda bo: pd.Series(assign_band((bo - 1) * 10 - 60 + 5)) if not pd.isna(bo) else pd.Series([np.nan, None])
)
# +1 HOUR shift is already baked into REAL_START_TIME by the SQL
df["REAL_START_TIME"] = pd.to_datetime(df["REAL_START_TIME"], format="%Y-%m-%d %H:%M")
# ── Exact payout terms (GGR FIX) ─────────────────────────────────────────────
# The SQL ships the EXACT weighted payout sums per row:
#   GGR_STAKE  = Σ(stake_eur × selection_coefficient)   (stake-weighted)
#   GGR_PLAYER = Σ(n_players × selection_coefficient)   (player-weighted)
# Use them directly; fall back to the old volume × mean-odds approximation
# only if the columns are missing.
if 'GGR_STAKE' in df.columns:
    df['STAKE_X_ODDS'] = pd.to_numeric(df['GGR_STAKE'], errors='coerce').fillna(0.0)
else:
    df['STAKE_X_ODDS'] = df['STAKE_EUR'] * df['ODDS']
if 'GGR_PLAYER' in df.columns:
    df['PLAYERS_X_ODDS'] = pd.to_numeric(df['GGR_PLAYER'], errors='coerce').fillna(0.0)
else:
    df['PLAYERS_X_ODDS'] = df['N_PLAYERS'] * df['ODDS']
df['TOURNAMENT_NAME'] = df['TOURNAMENT_NAME'].fillna('UNKNOWN')
# Keep only rows with valid odds (> 1)
df = df[df['ODDS'] > 1].copy()
# ─── STEP 2: Aggregate per event × band × outcome (data is already per outcome×band) ──
group_cols = ['EVENT_NAME', 'SPORT_NAME', 'TOURNAMENT_NAME', 'REAL_START_TIME', 'BAND_ORDER', 'BAND_LABEL']

# ── LAST-OBSERVED ODDS (CHANGE): preserve chronological order inside each band
#    so 'last' = last observed price, not a band average. Uses an acceptance-time
#    column if the SQL provides one; otherwise falls back to input row order.
_time_col = next((c for c in ['ACCEPTED_DT', 'ACCEPTED_TS', 'ACCEPTED_DATETIME', 'ACCEPTED_TIME']
                  if c in df.columns), None)
if _time_col:
    df = df.sort_values(group_cols + ['OUTCOME', _time_col], kind='stable')

band_agg = df.groupby(group_cols + ['OUTCOME'], dropna=False).agg(
    STAKE_SIGNAL   = ('STAKE_EUR',      'sum'),
    STAKE_X_ODDS   = ('STAKE_X_ODDS',   'sum'),   # exact Σ(stake × odds)
    PLAYERS_X_ODDS = ('PLAYERS_X_ODDS', 'sum'),   # exact Σ(players × odds)
    N_TICKETS      = ('N_PLAYERS',      'sum'),
    N_PLAYERS      = ('N_PLAYERS',      'sum'),
    AVG_ODDS       = ('ODDS',           'mean'),  # display only — NOT used in GGR
    LAST_ODDS_OBS  = ('ODDS',           'last')   # last observed price in the band (CHANGE)
).reset_index()
# ─── STEP 3: Pivot to one row per event × band ──────────────────────────────
pivoted = band_agg.pivot_table(
    index=group_cols,
    columns='OUTCOME',
    values=['STAKE_SIGNAL', 'STAKE_X_ODDS', 'PLAYERS_X_ODDS', 'N_TICKETS', 'N_PLAYERS', 'AVG_ODDS', 'LAST_ODDS_OBS'],
    aggfunc='first',
    fill_value=0
).reset_index()
pivoted.columns = ['_'.join(col).strip('_') if col[1] else col[0] for col in pivoted.columns]
rename_map = {
    'STAKE_SIGNAL_1':   'STAKE_1',  'STAKE_SIGNAL_X':   'STAKE_X',  'STAKE_SIGNAL_2':   'STAKE_2',
    'STAKE_X_ODDS_1':   'SXO_1',    'STAKE_X_ODDS_X':   'SXO_X',    'STAKE_X_ODDS_2':   'SXO_2',
    'PLAYERS_X_ODDS_1': 'PXO_1',    'PLAYERS_X_ODDS_X': 'PXO_X',    'PLAYERS_X_ODDS_2': 'PXO_2',
    'N_TICKETS_1':      'NT_1',     'N_TICKETS_X':      'NT_X',     'N_TICKETS_2':      'NT_2',
    'N_PLAYERS_1':      'NP_1',     'N_PLAYERS_X':      'NP_X',     'N_PLAYERS_2':      'NP_2',
    'AVG_ODDS_1':       'ODD_1',    'AVG_ODDS_X':       'ODD_X',    'AVG_ODDS_2':       'ODD_2',
    'LAST_ODDS_OBS_1':  'LODD_1',   'LAST_ODDS_OBS_X':  'LODD_X',   'LAST_ODDS_OBS_2':  'LODD_2',
}
pivoted = pivoted.rename(columns=rename_map)
for col in ['STAKE_1','STAKE_X','STAKE_2',
            'SXO_1','SXO_X','SXO_2',
            'PXO_1','PXO_X','PXO_2',
            'NT_1','NT_X','NT_2',
            'NP_1','NP_X','NP_2',
            'ODD_1','ODD_X','ODD_2',
            'LODD_1','LODD_X','LODD_2']:
    if col not in pivoted.columns:
        pivoted[col] = 0
# ─── STEP 3b: Determine 2-way vs 3-way sports ─────────────────────────────
two_way_sports = ['Basketball', 'Tennis', 'Volleyball']
is_two_way = pivoted['SPORT_NAME'].isin(two_way_sports)
pivoted.loc[is_two_way, ['STAKE_X', 'SXO_X', 'PXO_X', 'NT_X', 'NP_X', 'ODD_X', 'LODD_X']] = 0
# ─── STEP 3c: Compute GGR per selection — EXACT weighted payout (GGR FIX) ────
# OLD (biased):  GGR_o       = NT_GAME    − NT_o    × ODD_o
#                GGR_STAKE_o = STAKE_GAME − STAKE_o × ODD_o
#                (payout ≈ volume × unweighted mean of per-grain mean odds)
# NEW (exact):   payout = stake/player-WEIGHTED sum of odds (SXO_o / PXO_o)
pivoted['NT_GAME']    = pivoted['NT_1']    + pivoted['NT_X']    + pivoted['NT_2']
pivoted['STAKE_GAME'] = pivoted['STAKE_1'] + pivoted['STAKE_X'] + pivoted['STAKE_2']
for o in ['1', 'X', '2']:
    pivoted[f'GGR_{o}']       = pivoted['NT_GAME']    - pivoted[f'PXO_{o}']
    pivoted[f'GGR_STAKE_{o}'] = pivoted['STAKE_GAME'] - pivoted[f'SXO_{o}']
pivoted.loc[is_two_way, ['GGR_X', 'GGR_STAKE_X']] = 0
# ─── STEP 4: Whole-game totals ───────────────────────────────────────────
pivoted['NP_GAME'] = pivoted['NP_1'] + pivoted['NP_X'] + pivoted['NP_2']
pivoted['STAKE_TOTAL_GAME'] = pivoted.groupby('EVENT_NAME')['STAKE_GAME'].transform('sum')
pivoted['NP_TOTAL_GAME']    = pivoted.groupby('EVENT_NAME')['NP_GAME'].transform('sum')
# ─── STEP 5: Score + Softmax ─────────────────────────────────────────────
for o in ['1', 'X', '2']:
    pivoted[f'SCORE_STAKE_{o}'] = np.where(
        pivoted['STAKE_TOTAL_GAME'] > 0, pivoted[f'GGR_STAKE_{o}'] / pivoted['STAKE_TOTAL_GAME'], 0)
    pivoted[f'SCORE_PLAYER_{o}'] = np.where(
        pivoted['NP_TOTAL_GAME'] > 0, pivoted[f'GGR_{o}'] / pivoted['NP_TOTAL_GAME'], 0)
pivoted.loc[is_two_way, 'SCORE_PLAYER_X'] = -1e9
pivoted.loc[is_two_way, 'SCORE_STAKE_X']  = -1e9
pivoted['SCORE_PLAYER_MAX'] = pivoted[['SCORE_PLAYER_1', 'SCORE_PLAYER_X', 'SCORE_PLAYER_2']].max(axis=1)
pivoted['SCORE_STAKE_MAX']  = pivoted[['SCORE_STAKE_1',  'SCORE_STAKE_X',  'SCORE_STAKE_2']].max(axis=1)
for o in ['1', 'X', '2']:
    pivoted[f'EXP_PLAYER_{o}'] = np.exp(pivoted[f'SCORE_PLAYER_{o}'] - pivoted['SCORE_PLAYER_MAX'])
    pivoted[f'EXP_STAKE_{o}']  = np.exp(pivoted[f'SCORE_STAKE_{o}']  - pivoted['SCORE_STAKE_MAX'])
pivoted.loc[is_two_way, 'EXP_PLAYER_X'] = 0
pivoted.loc[is_two_way, 'EXP_STAKE_X']  = 0
exp_sum_player = pivoted['EXP_PLAYER_1'] + pivoted['EXP_PLAYER_X'] + pivoted['EXP_PLAYER_2']
exp_sum_stake  = pivoted['EXP_STAKE_1']  + pivoted['EXP_STAKE_X']  + pivoted['EXP_STAKE_2']
for o in ['1', 'X', '2']:
    pivoted[f'P_PLAYER_{o}'] = pivoted[f'EXP_PLAYER_{o}'] / exp_sum_player
    pivoted[f'P_STAKE_{o}']  = pivoted[f'EXP_STAKE_{o}']  / exp_sum_stake
# ─── STEP 6: Cumulative Bayesian update ──────────────────────────────────
PLAYER_GGR_WEIGHT = 0.33
STAKE_GGR_WEIGHT  = 0.66
pivoted = pivoted.sort_values(group_cols)
pivoted.loc[pivoted['SPORT_NAME'].isin(two_way_sports), 'P_PLAYER_X'] = 1.0
pivoted.loc[pivoted['SPORT_NAME'].isin(two_way_sports), 'P_STAKE_X']  = 1.0
for track in ['PLAYER', 'STAKE']:
    for o in ['1', 'X', '2']:
        pivoted[f'LOG_CUM_{track}_{o}'] = pivoted.groupby('EVENT_NAME')[f'P_{track}_{o}'].transform(
            lambda x: np.log(np.maximum(x, 1e-10)).cumsum())
    pivoted.loc[pivoted['SPORT_NAME'].isin(two_way_sports), f'LOG_CUM_{track}_X'] = -1e9
    pivoted[f'LOG_CUM_{track}_MAX'] = pivoted[[f'LOG_CUM_{track}_1', f'LOG_CUM_{track}_X', f'LOG_CUM_{track}_2']].max(axis=1)
    for o in ['1', 'X', '2']:
        pivoted[f'ECUM_{track}_{o}'] = np.exp(pivoted[f'LOG_CUM_{track}_{o}'] - pivoted[f'LOG_CUM_{track}_MAX'])
    pivoted.loc[pivoted['SPORT_NAME'].isin(two_way_sports), f'ECUM_{track}_X'] = 0
    ecum_sum = pivoted[f'ECUM_{track}_1'] + pivoted[f'ECUM_{track}_X'] + pivoted[f'ECUM_{track}_2']
    for o in ['1', 'X', '2']:
        pivoted[f'CUM_P_{track}_{o}'] = (pivoted[f'ECUM_{track}_{o}'] / ecum_sum).round(4)
for o in ['1', 'X', '2']:
    pivoted[f'P_{o}']     = (PLAYER_GGR_WEIGHT * pivoted[f'P_PLAYER_{o}']     + STAKE_GGR_WEIGHT * pivoted[f'P_STAKE_{o}']).round(4)
    pivoted[f'CUM_P_{o}'] = (PLAYER_GGR_WEIGHT * pivoted[f'CUM_P_PLAYER_{o}'] + STAKE_GGR_WEIGHT * pivoted[f'CUM_P_STAKE_{o}']).round(4)
pivoted.loc[pivoted['SPORT_NAME'].isin(two_way_sports), 'P_X'] = 1.0
# ─── STEP 7: Prediction & Confidence ────────────────────────────────────
is_two_way_final = pivoted['SPORT_NAME'].isin(two_way_sports)
pivoted.loc[~is_two_way_final, 'PRED_RESULT'] = pivoted.loc[~is_two_way_final, ['CUM_P_1','CUM_P_X','CUM_P_2']].idxmax(axis=1).str[-1]
pivoted.loc[~is_two_way_final, 'CONFIDENCE'] = pivoted.loc[~is_two_way_final, ['CUM_P_1','CUM_P_X','CUM_P_2']].max(axis=1).round(4)
pivoted.loc[~is_two_way_final, 'ENTROPY'] = -(
      pivoted.loc[~is_two_way_final, 'CUM_P_1'] * np.log(np.maximum(pivoted.loc[~is_two_way_final, 'CUM_P_1'], 1e-10))
    + pivoted.loc[~is_two_way_final, 'CUM_P_X'] * np.log(np.maximum(pivoted.loc[~is_two_way_final, 'CUM_P_X'], 1e-10))
    + pivoted.loc[~is_two_way_final, 'CUM_P_2'] * np.log(np.maximum(pivoted.loc[~is_two_way_final, 'CUM_P_2'], 1e-10))
).round(4)
pivoted.loc[is_two_way_final, 'PRED_RESULT'] = pivoted.loc[is_two_way_final, ['CUM_P_1','CUM_P_2']].idxmax(axis=1).str[-1]
pivoted.loc[is_two_way_final, 'CONFIDENCE'] = pivoted.loc[is_two_way_final, ['CUM_P_1','CUM_P_2']].max(axis=1).round(4)
pivoted.loc[is_two_way_final, 'ENTROPY'] = -(
      pivoted.loc[is_two_way_final, 'CUM_P_1'] * np.log(np.maximum(pivoted.loc[is_two_way_final, 'CUM_P_1'], 1e-10))
    + pivoted.loc[is_two_way_final, 'CUM_P_2'] * np.log(np.maximum(pivoted.loc[is_two_way_final, 'CUM_P_2'], 1e-10))
).round(4)
# ─── STEP 8: Snapshot builder (same as Model Notebook) ───────────────────
final_detail = pivoted[
    group_cols
    + ['GGR_1','GGR_X','GGR_2','GGR_STAKE_1','GGR_STAKE_X','GGR_STAKE_2',
       'STAKE_1','STAKE_X','STAKE_2','SXO_1','SXO_X','SXO_2',
       'NT_1','NT_X','NT_2','NT_GAME','NP_1','NP_X','NP_2',
       'ODD_1','ODD_X','ODD_2','LODD_1','LODD_X','LODD_2','P_1','P_X','P_2',
       'CUM_P_1','CUM_P_X','CUM_P_2',
       'CUM_P_STAKE_1','CUM_P_STAKE_X','CUM_P_STAKE_2',
       'CUM_P_PLAYER_1','CUM_P_PLAYER_X','CUM_P_PLAYER_2',
       'PRED_RESULT','CONFIDENCE','ENTROPY']
].copy()
# Pre-game bands (1-5) included so events before kickoff still get metrics
SNAPSHOT_BANDS = [(1,'PRE60'),(2,'PRE50'),(3,'PRE40'),(4,'PRE30'),(5,'PRE20'),
                  (6,'KO'),(7,'M10'),(8,'M20'),(9,'M30'),(10,'M40'),(11,'M50'),
                  (12,'M60'),(13,'M70'),(14,'M80'),(15,'M90'),(16,'M100'),
                  (17,'M110'),(18,'M120'),(19,'M130'),(20,'M140'),(21,'M150')]
TREND_MA_WINDOW = 3
PRE_GAME_WEIGHT = 1
IN_GAME_WEIGHT  = 2
def _wavg_odds(bands_df, outcome):
    total_stake = bands_df[f'STAKE_{outcome}'].sum()
    if total_stake <= 0:
        return None
    return round(float(bands_df[f'SXO_{outcome}'].sum()) / float(total_stake), 4)
def build_snapshot(grp):
    row = {}
    sport = grp['SPORT_NAME'].iloc[0]
    row['EVENT_NAME'] = grp['EVENT_NAME'].iloc[0]
    row['SPORT_NAME'] = sport
    row['TOURNAMENT_NAME'] = grp['TOURNAMENT_NAME'].iloc[0]
    row['REAL_START_TIME'] = grp['REAL_START_TIME'].iloc[0]
    # NOTE: REAL_START_TIME already has the +1h SQL shift baked in.
    row['REAL_START_TIME_PLUS_1H'] = grp['REAL_START_TIME'].iloc[0].strftime('%Y-%m-%d %H:%M')
    row['ODD_1'] = grp['ODD_1'].max()
    row['ODD_X'] = grp['ODD_X'].max() if sport not in two_way_sports else None
    row['ODD_2'] = grp['ODD_2'].max()
    row['BANDS_WITH_DATA'] = int(grp['BAND_ORDER'].nunique())
    row['LAST_BAND'] = int(grp['BAND_ORDER'].max())
    # ── N players who bet in the -10 to +9 min kickoff window ──────────────
    # Band 6 = "−10 to −1 min", Band 7 = "0 to +9 min"
    ko_window = grp[grp['BAND_ORDER'].isin([6, 7])]
    row['N_PLAYERS_KO_WINDOW_10MIN'] = int(ko_window[['NP_1', 'NP_X', 'NP_2']].sum().sum())
    pre_match = grp[grp['BAND_ORDER'] <= 6]
    grp_sorted = grp.sort_values('BAND_ORDER')
    for _o in ['1', 'X', '2']:
        if _o == 'X' and sport in two_way_sports:
            row[f'AVG_ODDS_PRE_{_o}'] = None
            row[f'LAST_ODDS_{_o}'] = None
        else:
            pre_vals = pre_match[f'ODD_{_o}'].replace(0, np.nan).dropna()
            row[f'AVG_ODDS_PRE_{_o}'] = round(float(pre_vals.mean()), 2) if len(pre_vals) > 0 else None
            # LAST_ODDS (CHANGE): last OBSERVED price (LODD_*), not the band average —
            # the LODD value from the latest band with odds data for this outcome.
            last_vals = grp_sorted[f'LODD_{_o}'].replace(0, np.nan).dropna()
            row[f'LAST_ODDS_{_o}'] = round(float(last_vals.iloc[-1]), 4) if len(last_vals) > 0 else None
    # ── Odds drift ratio (NEW): DELTA_ODDS = LAST_ODDS / AVG_ODDS_PRE ───────
    # DELTA_ODDS > 1 → odds drifted out (outcome seen as less likely than pre-match)
    # DELTA_ODDS < 1 → odds shortened  (outcome seen as more likely)
    for _o in ['1', 'X', '2']:
        if _o == 'X' and sport in two_way_sports:
            row[f'DELTA_ODDS_{_o}'] = None
        else:
            _last = row.get(f'LAST_ODDS_{_o}')
            _pre  = row.get(f'AVG_ODDS_PRE_{_o}')
            row[f'DELTA_ODDS_{_o}'] = (
                round(float(_last) / float(_pre), 4)
                if (_last is not None and _pre is not None and float(_pre) > 0)
                else None
            )
    last_band = row['LAST_BAND']
    pre_bands = grp[grp['BAND_ORDER'] <= 6]
    inplay_bands = grp[grp['BAND_ORDER'] >= 7]
    for o in ['1', 'X', '2']:
        if o == 'X' and sport in two_way_sports:
            row[f'WAVG_ODD_PRE_{o}'] = None
            row[f'WAVG_ODD_INPLAY_{o}'] = None
        else:
            row[f'WAVG_ODD_PRE_{o}'] = _wavg_odds(pre_bands, o)
            row[f'WAVG_ODD_INPLAY_{o}'] = _wavg_odds(inplay_bands, o)
    # ── Post-kickoff aggregated GGR per outcome ─────────────────────────────
    # Sum of the per-band KPIs across all IN-PLAY bands only (BAND_ORDER >= 7,
    # i.e. from "0 to +9 min" onwards — the KO band 6 "−10 to −1 min" and all
    # earlier pre-match bands are excluded).
    for o in ['1', 'X', '2']:
        if o == 'X' and sport in two_way_sports:
            row[f'CUM_GGR_STAKE_POSTKO_{o}']  = 0.0
            row[f'CUM_GGR_PLAYER_POSTKO_{o}'] = 0.0
        else:
            row[f'CUM_GGR_STAKE_POSTKO_{o}']  = round(float(inplay_bands[f'GGR_STAKE_{o}'].sum()), 4) if len(inplay_bands) > 0 else None
            row[f'CUM_GGR_PLAYER_POSTKO_{o}'] = round(float(inplay_bands[f'GGR_{o}'].sum()), 4) if len(inplay_bands) > 0 else None
    for band, prefix in SNAPSHOT_BANDS:
        if band > last_band:
            break
        b = grp[grp['BAND_ORDER'] == band]
        bth = grp[grp['BAND_ORDER'] <= band]
        if len(b) > 0:
            row[f'{prefix}_PROB_1'] = b['CUM_P_1'].values[0]
            row[f'{prefix}_PROB_X'] = b['CUM_P_X'].values[0] if sport not in two_way_sports else None
            row[f'{prefix}_PROB_2'] = b['CUM_P_2'].values[0]
            row[f'{prefix}_PRED'] = b['PRED_RESULT'].values[0]
            row[f'{prefix}_CONFIDENCE'] = b['CONFIDENCE'].values[0]
            row[f'{prefix}_ENTROPY'] = b['ENTROPY'].values[0]
            _pred_o = b['PRED_RESULT'].values[0]
            row[f'{prefix}_CUM_P_STAKE_PRED'] = float(b[f'CUM_P_STAKE_{_pred_o}'].values[0])
            row[f'{prefix}_CUM_P_PLAYER_PRED'] = float(b[f'CUM_P_PLAYER_{_pred_o}'].values[0])
            p1b = float(b['P_1'].values[0])
            pXb = float(b['P_X'].values[0]) if sport not in two_way_sports else -1.0
            p2b = float(b['P_2'].values[0])
            if sport in two_way_sports:
                row[f'{prefix}_BAND_PRED'] = '1' if p1b >= p2b else '2'
            else:
                _bp = {'1': p1b, 'X': pXb, '2': p2b}
                row[f'{prefix}_BAND_PRED'] = max(_bp, key=_bp.get)
            row[f'{prefix}_CUM_GGR_PLAYER_1'] = round(float(bth['GGR_1'].sum()), 4)
            row[f'{prefix}_CUM_GGR_PLAYER_X'] = round(float(bth['GGR_X'].sum()), 4) if sport not in two_way_sports else 0.0
            row[f'{prefix}_CUM_GGR_PLAYER_2'] = round(float(bth['GGR_2'].sum()), 4)
            row[f'{prefix}_CUM_GGR_STAKE_1']  = round(float(bth['GGR_STAKE_1'].sum()), 4)
            row[f'{prefix}_CUM_GGR_STAKE_X']  = round(float(bth['GGR_STAKE_X'].sum()), 4) if sport not in two_way_sports else 0.0
            row[f'{prefix}_CUM_GGR_STAKE_2']  = round(float(bth['GGR_STAKE_2'].sum()), 4)
            row[f'{prefix}_GGR_STAKE_1'] = round(float(b['GGR_STAKE_1'].values[0]), 4)
            row[f'{prefix}_GGR_STAKE_X'] = round(float(b['GGR_STAKE_X'].values[0]), 4) if sport not in two_way_sports else 0.0
            row[f'{prefix}_GGR_STAKE_2'] = round(float(b['GGR_STAKE_2'].values[0]), 4)
            row[f'{prefix}_GGR_PLAYER_1'] = round(float(b['GGR_1'].values[0]), 4)
            row[f'{prefix}_GGR_PLAYER_X'] = round(float(b['GGR_X'].values[0]), 4) if sport not in two_way_sports else 0.0
            row[f'{prefix}_GGR_PLAYER_2'] = round(float(b['GGR_2'].values[0]), 4)
            row[f'{prefix}_CUM_N_PLAYERS'] = int(bth[['NP_1','NP_X','NP_2']].sum().sum())
        else:
            for col in ['PROB_1','PROB_X','PROB_2','PRED','CONFIDENCE','ENTROPY','BAND_PRED',
                        'CUM_GGR_PLAYER_1','CUM_GGR_PLAYER_X','CUM_GGR_PLAYER_2',
                        'CUM_GGR_STAKE_1','CUM_GGR_STAKE_X','CUM_GGR_STAKE_2',
                        'GGR_STAKE_1','GGR_STAKE_X','GGR_STAKE_2',
                        'GGR_PLAYER_1','GGR_PLAYER_X','GGR_PLAYER_2',
                        'CUM_N_PLAYERS','CUM_P_STAKE_PRED','CUM_P_PLAYER_PRED']:
                row[f'{prefix}_{col}'] = None
    # Confidence deltas (includes pre-game to KO transitions)
    delta_pairs = [('PRE60','PRE50','CONF_DELTA_PRE60_TO_PRE50'),('PRE50','PRE40','CONF_DELTA_PRE50_TO_PRE40'),
        ('PRE40','PRE30','CONF_DELTA_PRE40_TO_PRE30'),('PRE30','PRE20','CONF_DELTA_PRE30_TO_PRE20'),
        ('PRE20','KO','CONF_DELTA_PRE20_TO_KO'),
        ('KO','M10','CONF_DELTA_KO_TO_M10'),('M10','M20','CONF_DELTA_M10_TO_M20'),
        ('M20','M30','CONF_DELTA_M20_TO_M30'),('M30','M40','CONF_DELTA_M30_TO_M40'),
        ('M40','M50','CONF_DELTA_M40_TO_M50'),('M50','M60','CONF_DELTA_M50_TO_M60'),
        ('M60','M70','CONF_DELTA_M60_TO_M70'),('M70','M80','CONF_DELTA_M70_TO_M80'),
        ('M80','M90','CONF_DELTA_M80_TO_M90'),('M90','M100','CONF_DELTA_M90_TO_M100'),
        ('M100','M110','CONF_DELTA_M100_TO_M110'),('M110','M120','CONF_DELTA_M110_TO_M120'),
        ('M120','M130','CONF_DELTA_M120_TO_M130'),('M130','M140','CONF_DELTA_M130_TO_M140'),
        ('M140','M150','CONF_DELTA_M140_TO_M150')]
    for prev_pfx, next_pfx, delta_name in delta_pairs:
        prev_c = row.get(f'{prev_pfx}_CONFIDENCE')
        next_c = row.get(f'{next_pfx}_CONFIDENCE')
        row[delta_name] = round(next_c - prev_c, 4) if (prev_c is not None and next_c is not None) else None
    # Last available snapshot
    last_conf = None
    for _, prefix in reversed(SNAPSHOT_BANDS):
        c = row.get(f'{prefix}_CONFIDENCE')
        if c is not None:
            last_conf = c
            row['LAST_SNAPSHOT'] = prefix
            break
    ko_conf = row.get('KO_CONFIDENCE')
    row['CONF_DELTA_KO_TO_M_LAST_SNAPSHOT'] = round(last_conf - ko_conf, 4) if (last_conf is not None and ko_conf is not None) else None
    # M_LAST_SNAPSHOT_*
    pfx = row.get('LAST_SNAPSHOT')
    if pfx:
        row['M_LAST_SNAPSHOT_PROB_1'] = row.get(f'{pfx}_PROB_1')
        row['M_LAST_SNAPSHOT_PROB_X'] = row.get(f'{pfx}_PROB_X')
        row['M_LAST_SNAPSHOT_PROB_2'] = row.get(f'{pfx}_PROB_2')
        row['M_LAST_SNAPSHOT_PRED'] = row.get(f'{pfx}_PRED')
        row['M_LAST_SNAPSHOT_CONFIDENCE'] = row.get(f'{pfx}_CONFIDENCE')
        row['M_LAST_SNAPSHOT_ENTROPY'] = row.get(f'{pfx}_ENTROPY')
        row['M_LAST_SNAPSHOT_CUM_GGR_STAKE_1'] = row.get(f'{pfx}_CUM_GGR_STAKE_1')
        row['M_LAST_SNAPSHOT_CUM_GGR_STAKE_X'] = row.get(f'{pfx}_CUM_GGR_STAKE_X')
        row['M_LAST_SNAPSHOT_CUM_GGR_STAKE_2'] = row.get(f'{pfx}_CUM_GGR_STAKE_2')
        row['M_LAST_SNAPSHOT_CUM_GGR_PLAYER_1'] = row.get(f'{pfx}_CUM_GGR_PLAYER_1')
        row['M_LAST_SNAPSHOT_CUM_GGR_PLAYER_X'] = row.get(f'{pfx}_CUM_GGR_PLAYER_X')
        row['M_LAST_SNAPSHOT_CUM_GGR_PLAYER_2'] = row.get(f'{pfx}_CUM_GGR_PLAYER_2')
        row['M_LAST_SNAPSHOT_CUM_N_PLAYERS'] = row.get(f'{pfx}_CUM_N_PLAYERS')
        pred_outcome = row.get('M_LAST_SNAPSHOT_PRED')
        row['M_LAST_SNAPSHOT_PRED_PROB'] = row.get(f'M_LAST_SNAPSHOT_PROB_{pred_outcome}') if pred_outcome else None
        row['M_LAST_SNAPSHOT_PROB_STAKE'] = row.get(f'{pfx}_CUM_P_STAKE_PRED')
        row['M_LAST_SNAPSHOT_PROB_PLAYER'] = row.get(f'{pfx}_CUM_P_PLAYER_PRED')
    else:
        for col in ['M_LAST_SNAPSHOT_PROB_1','M_LAST_SNAPSHOT_PROB_X','M_LAST_SNAPSHOT_PROB_2',
                    'M_LAST_SNAPSHOT_PRED','M_LAST_SNAPSHOT_CONFIDENCE','M_LAST_SNAPSHOT_ENTROPY',
                    'M_LAST_SNAPSHOT_CUM_GGR_STAKE_1','M_LAST_SNAPSHOT_CUM_GGR_STAKE_X','M_LAST_SNAPSHOT_CUM_GGR_STAKE_2',
                    'M_LAST_SNAPSHOT_CUM_GGR_PLAYER_1','M_LAST_SNAPSHOT_CUM_GGR_PLAYER_X','M_LAST_SNAPSHOT_CUM_GGR_PLAYER_2',
                    'M_LAST_SNAPSHOT_CUM_N_PLAYERS','M_LAST_SNAPSHOT_PRED_PROB',
                    'M_LAST_SNAPSHOT_PROB_STAKE','M_LAST_SNAPSHOT_PROB_PLAYER']:
            row[col] = None
    # Signal consistency
    final_pred = row.get('M_LAST_SNAPSHOT_PRED')
    weighted_agree = 0.0
    weighted_total = 0.0
    for band, pfx in SNAPSHOT_BANDS:
        band_pred = row.get(f'{pfx}_BAND_PRED')
        if band_pred is None: continue
        # Pre-game bands (1-5) use PRE_GAME_WEIGHT, KO (6) uses PRE_GAME_WEIGHT, in-play (7+) uses IN_GAME_WEIGHT
        w = PRE_GAME_WEIGHT if band <= 6 else IN_GAME_WEIGHT
        weighted_total += w
        if final_pred is not None and band_pred == final_pred:
            weighted_agree += w
    row['SIGNAL_CONSISTENCY_PCT'] = round(weighted_agree / weighted_total * 100, 1) if (final_pred and weighted_total > 0) else None
    # Prediction stability
    preds = [row.get(f'{p}_PRED') for _, p in SNAPSHOT_BANDS]
    valid_preds = [p for p in preds if p is not None]
    if len(valid_preds) < 2:
        row['PRED_STABILITY'] = 'PARTIAL'
    elif len(set(valid_preds)) == 1:
        row['PRED_STABILITY'] = 'STABLE'
    else:
        row['PRED_STABILITY'] = 'FLIPPED'
    # Signal trend
    kpi_series = []
    for _, pfx in SNAPSHOT_BANDS:
        band_pred = row.get(f'{pfx}_BAND_PRED')
        if band_pred is None: continue
        ggr_player = row.get(f'{pfx}_GGR_PLAYER_{band_pred}')
        ggr_stake = row.get(f'{pfx}_GGR_STAKE_{band_pred}')
        if ggr_player is None or ggr_stake is None: continue
        kpi_series.append(PLAYER_GGR_WEIGHT * ggr_player + STAKE_GGR_WEIGHT * ggr_stake)
    if len(kpi_series) < 2:
        row['SIGNAL_TREND'] = None
    else:
        kpi_ma = pd.Series(kpi_series).rolling(window=min(TREND_MA_WINDOW, len(kpi_series)), min_periods=1).mean().values
        valid_ma = kpi_ma[~np.isnan(kpi_ma)]
        if len(valid_ma) < 2:
            row['SIGNAL_TREND'] = None
        else:
            slope = np.polyfit(np.arange(len(valid_ma), dtype=float), valid_ma, 1)[0]
            if abs(slope) < 0.003:
                row['SIGNAL_TREND'] = 'FLAT'
            else:
                direction = 'INCREASING' if slope > 0 else 'DECREASING'
                strength = 'STRONG' if abs(slope) >= 0.02 else ('MODERATE' if abs(slope) >= 0.008 else 'LOW')
                row['SIGNAL_TREND'] = f'{direction}_{strength}'
    return row
# ── Build result table ────────────────────────────────────────────────────
result_rows = []
for _, grp in final_detail.groupby('EVENT_NAME', dropna=False):
    result_rows.append(build_snapshot(grp))
result = pd.DataFrame(result_rows)
result = result.sort_values(['BANDS_WITH_DATA', 'LAST_BAND'], ascending=[False, False]).reset_index(drop=True)
# ─── STEP 9: Tournament history comparison ───────────────────────────────
try:
    _hist = df_tournament_history.copy()
    _hist['TOURNAMENT_NAME'] = _hist['TOURNAMENT_NAME'].where(
        ~_hist['TOURNAMENT_NAME'].str.contains('World Cup 2026', case=False, na=False), other='World Cup 2026')
    ref_rows = []
    for idx, r in result.iterrows():
        tourn, last_b, pred = r.get('TOURNAMENT_NAME'), r.get('LAST_BAND'), r.get('M_LAST_SNAPSHOT_PRED')
        if pd.isna(tourn) or tourn == 'UNKNOWN' or pd.isna(last_b) or pd.isna(pred):
            ref_rows.append({'REF_MAX_GGR_EUR': None, 'REF_MAX_CUM_N_PLAYERS': None}); continue
        mask = (_hist['TOURNAMENT_NAME'] == tourn) & (_hist['BAND_ORDER'] == int(last_b)) & (_hist['OUTCOME'] == str(pred))
        matched = _hist.loc[mask]
        if len(matched) > 0:
            ref_rows.append({'REF_MAX_GGR_EUR': float(matched['MAX_CUM_GGR_EUR_STAKE'].iloc[0]),
                             'REF_MAX_CUM_N_PLAYERS': int(matched['MAX_CUM_N_PLAYERS'].iloc[0])})
        else:
            ref_rows.append({'REF_MAX_GGR_EUR': None, 'REF_MAX_CUM_N_PLAYERS': None})
    ref_df = pd.DataFrame(ref_rows)
    result['REF_MAX_GGR_EUR'] = pd.to_numeric(ref_df['REF_MAX_GGR_EUR'], errors='coerce').values
    result['REF_MAX_CUM_N_PLAYERS'] = pd.to_numeric(ref_df['REF_MAX_CUM_N_PLAYERS'], errors='coerce').values
    def _get_last_cum_ggr_stake(r):
        pfx, pred = r.get('LAST_SNAPSHOT'), r.get('M_LAST_SNAPSHOT_PRED')
        return r.get(f'{pfx}_CUM_GGR_STAKE_{pred}') if pfx and pred else None
    result['_CUM_GGR_STAKE_LAST'] = pd.to_numeric(result.apply(_get_last_cum_ggr_stake, axis=1), errors='coerce')
    result['_CUM_N_PLAYERS_LAST'] = pd.to_numeric(result.apply(
        lambda r: r.get(f"{r.get('LAST_SNAPSHOT')}_CUM_N_PLAYERS") if r.get('LAST_SNAPSHOT') else None, axis=1), errors='coerce')
    result['GGR_STAKE_PCT_OF_MAX'] = np.where(
        result['REF_MAX_GGR_EUR'].notna() & (result['REF_MAX_GGR_EUR'].abs() > 0),
        (result['_CUM_GGR_STAKE_LAST'].abs() / result['REF_MAX_GGR_EUR'].abs() * 100).round(1), None)
    result['GGR_PLAYER_PCT_OF_MAX'] = np.where(
        result['REF_MAX_CUM_N_PLAYERS'].notna() & (result['REF_MAX_CUM_N_PLAYERS'] > 0),
        (result['_CUM_N_PLAYERS_LAST'] / result['REF_MAX_CUM_N_PLAYERS'] * 100).round(1), None)
    # NOTE: '_CUM_GGR_STAKE_LAST' and '_CUM_N_PLAYERS_LAST' are kept (not dropped
    # here) because STEP 9b/9c reuse them to build the STAKE-based damping
    # ratios. They are dropped once, at the very end of STEP 9c.
except NameError:
    result['REF_MAX_GGR_EUR'] = None
    result['REF_MAX_CUM_N_PLAYERS'] = None
    result['GGR_STAKE_PCT_OF_MAX'] = None
    result['GGR_PLAYER_PCT_OF_MAX'] = None
    result['_CUM_GGR_STAKE_LAST'] = None
    result['_CUM_N_PLAYERS_LAST'] = None
# ─── STEP 9b: Tournament-context damping (PLAYER + STAKE) ─────────────────
def _get_last_cum_ggr_player(r):
    pfx, pred = r.get('LAST_SNAPSHOT'), r.get('M_LAST_SNAPSHOT_PRED')
    return r.get(f'{pfx}_CUM_GGR_PLAYER_{pred}') if pfx and pred else None
result['_CUM_GGR_PLAYER_LAST'] = pd.to_numeric(result.apply(_get_last_cum_ggr_player, axis=1), errors='coerce')
result['REF_MAX_CUM_N_PLAYERS'] = pd.to_numeric(result['REF_MAX_CUM_N_PLAYERS'], errors='coerce')
result['REF_MAX_GGR_EUR'] = pd.to_numeric(result['REF_MAX_GGR_EUR'], errors='coerce')
result['M_LAST_SNAPSHOT_PRED_PROB'] = pd.to_numeric(result['M_LAST_SNAPSHOT_PRED_PROB'], errors='coerce')
result['_CUM_GGR_STAKE_LAST'] = pd.to_numeric(result['_CUM_GGR_STAKE_LAST'], errors='coerce')

# --- PLAYER-based ratio ---
result['RATIO_GGR_PLAYER_TO_TOURNAMENT_MAX'] = np.where(
    result['REF_MAX_CUM_N_PLAYERS'].notna() & (result['REF_MAX_CUM_N_PLAYERS'] > 0) & result['_CUM_GGR_PLAYER_LAST'].notna(),
    (result['_CUM_GGR_PLAYER_LAST'].abs() / result['REF_MAX_CUM_N_PLAYERS']), np.nan)
result['RATIO_GGR_PLAYER_TO_TOURNAMENT_MAX'] = result['RATIO_GGR_PLAYER_TO_TOURNAMENT_MAX'].astype(float).round(4)
result['M_LAST_SNAPSHOT_PRED_PROB_ADJUSTED'] = np.where(
    result['M_LAST_SNAPSHOT_PRED_PROB'].notna() & result['RATIO_GGR_PLAYER_TO_TOURNAMENT_MAX'].notna(),
    (result['RATIO_GGR_PLAYER_TO_TOURNAMENT_MAX'] * result['M_LAST_SNAPSHOT_PRED_PROB']), np.nan)
result['M_LAST_SNAPSHOT_PRED_PROB_ADJUSTED'] = result['M_LAST_SNAPSHOT_PRED_PROB_ADJUSTED'].astype(float).round(4)

# --- STAKE-based ratio ---
result['RATIO_GGR_STAKE_TO_TOURNAMENT_MAX'] = np.where(
    result['REF_MAX_GGR_EUR'].notna() & (result['REF_MAX_GGR_EUR'].abs() > 0) & result['_CUM_GGR_STAKE_LAST'].notna(),
    (result['_CUM_GGR_STAKE_LAST'].abs() / result['REF_MAX_GGR_EUR'].abs()), np.nan)
result['RATIO_GGR_STAKE_TO_TOURNAMENT_MAX'] = result['RATIO_GGR_STAKE_TO_TOURNAMENT_MAX'].astype(float).round(4)
result['M_LAST_SNAPSHOT_PRED_PROB_ADJUSTED_STAKE'] = np.where(
    result['M_LAST_SNAPSHOT_PRED_PROB'].notna() & result['RATIO_GGR_STAKE_TO_TOURNAMENT_MAX'].notna(),
    (result['RATIO_GGR_STAKE_TO_TOURNAMENT_MAX'] * result['M_LAST_SNAPSHOT_PRED_PROB']), np.nan)
result['M_LAST_SNAPSHOT_PRED_PROB_ADJUSTED_STAKE'] = result['M_LAST_SNAPSHOT_PRED_PROB_ADJUSTED_STAKE'].astype(float).round(4)
# ─── STEP 9c: Team-context damping (PLAYER + STAKE) — AMENDED PLAYER TRACK ──
# CHANGE vs previous version:
#   REF_MAX_CUM_N_PLAYERS_TEAM (average of the two teams' MAX_CUM_N_PLAYERS)
#   is REPLACED by REF_MIN_GGR_PLAYER_TEAM:
#     • anchored to the team owning the CURRENT weakest STAKE-GGR side
#       (LOWEST_GGR_STAKE_OUTCOME: '1' → team1, '2' → team2, 'X' → lower of both)
#     • looked up in df_team_history['MAX_CUM_N_PLAYERS'] at BAND_ORDER = LAST_BAND
#     • .min() across ALL matched rows (not .iloc[0]), used in absolute value
#   The PLAYER numerator likewise switches from the predicted outcome to the
#   weakest side: _CUM_GGR_PLAYER_LOWEST = |{LAST_SNAPSHOT}_CUM_GGR_PLAYER_{low}|.

def _split_teams(event_name):
    if pd.isna(event_name) or '\u00b7' not in str(event_name):
        return None, None
    parts = str(event_name).split('\u00b7')
    return (parts[0].strip(), parts[1].strip()) if len(parts) == 2 else (None, None)

# Identify the outcome with the LOWEST current cumulative GGR_STAKE at the last
# snapshot (drives BOTH the stake track and, now, the player track).
def _lowest_ggr_stake_outcome(r):
    pfx = r.get('LAST_SNAPSHOT')
    if not pfx:
        return None
    sport = r.get('SPORT_NAME')
    outcomes = ['1', '2'] if sport in two_way_sports else ['1', 'X', '2']
    candidates = {}
    for o in outcomes:
        v = r.get(f'{pfx}_CUM_GGR_STAKE_{o}')
        if v is not None and not pd.isna(v):
            candidates[o] = float(v)
    if not candidates:
        return None
    return min(candidates, key=candidates.get)

result['LOWEST_GGR_STAKE_OUTCOME'] = result.apply(_lowest_ggr_stake_outcome, axis=1)

try:
    _team_hist = df_team_history.copy()
    _has_team_stake_max_col  = 'MAX_CUM_GGR_EUR_STAKE' in _team_hist.columns
    _has_team_stake_min_col  = 'MIN_CUM_GGR_EUR_STAKE' in _team_hist.columns
    _has_team_player_col     = 'MAX_CUM_N_PLAYERS'     in _team_hist.columns
    team_ref_rows = []
    for idx, r in result.iterrows():
        event_name, last_b = r.get('EVENT_NAME'), r.get('LAST_BAND')
        team1, team2 = _split_teams(event_name)
        if team1 is None or team2 is None or pd.isna(last_b):
            team_ref_rows.append({
                'REF_MIN_GGR_PLAYER_TEAM': None,
                'REF_MAX_GGR_EUR_TEAM': None,
                'REF_MIN_GGR_EUR_TEAM': None,
            })
            continue
        band_mask = _team_hist['BAND_ORDER'] == int(last_b)
        low_outcome = r.get('LOWEST_GGR_STAKE_OUTCOME')

        # ── REF_MIN_GGR_PLAYER_TEAM (NEW): historical player-GGR reference of
        #    the team owning the current weakest STAKE-GGR side, from
        #    MAX_CUM_N_PLAYERS at this band; .min() across all matched rows.
        ref_min_player_team = None
        if _has_team_player_col:
            def _team_min_player(team):
                m = pd.to_numeric(
                    _team_hist.loc[(_team_hist['TEAM'] == team) & band_mask, 'MAX_CUM_N_PLAYERS'],
                    errors='coerce'
                ).dropna()
                return float(m.min()) if len(m) > 0 else None
            if low_outcome == '1':
                ref_min_player_team = _team_min_player(team1)
            elif low_outcome == '2':
                ref_min_player_team = _team_min_player(team2)
            elif low_outcome == 'X':
                mvals = [v for v in (_team_min_player(team1), _team_min_player(team2)) if v is not None]
                ref_min_player_team = min(mvals) if mvals else None

        # ── REF_MAX_GGR_EUR_TEAM (unchanged): avg of the two teams' MAX stake-GGR
        ref_max_stake_team = None
        if _has_team_stake_max_col:
            s1 = _team_hist.loc[(_team_hist['TEAM'] == team1) & band_mask, 'MAX_CUM_GGR_EUR_STAKE']
            s2 = _team_hist.loc[(_team_hist['TEAM'] == team2) & band_mask, 'MAX_CUM_GGR_EUR_STAKE']
            svals = [float(s.iloc[0]) for s in [s1, s2] if len(s) > 0]
            ref_max_stake_team = (sum(svals) / len(svals)) if svals else None

        # ── REF_MIN_GGR_EUR_TEAM (unchanged): weakest side's historical MIN
        #    cumulative GGR_STAKE at this band, .min() across all matched rows.
        ref_min_stake_team = None
        if _has_team_stake_min_col:
            def _team_min_stake(team):
                m = pd.to_numeric(
                    _team_hist.loc[(_team_hist['TEAM'] == team) & band_mask, 'MIN_CUM_GGR_EUR_STAKE'],
                    errors='coerce'
                ).dropna()
                return float(m.min()) if len(m) > 0 else None
            if low_outcome == '1':
                ref_min_stake_team = _team_min_stake(team1)
            elif low_outcome == '2':
                ref_min_stake_team = _team_min_stake(team2)
            elif low_outcome == 'X':
                mvals = [v for v in (_team_min_stake(team1), _team_min_stake(team2)) if v is not None]
                ref_min_stake_team = min(mvals) if mvals else None

        team_ref_rows.append({
            'REF_MIN_GGR_PLAYER_TEAM': ref_min_player_team,
            'REF_MAX_GGR_EUR_TEAM': ref_max_stake_team,
            'REF_MIN_GGR_EUR_TEAM': ref_min_stake_team,
        })
    _team_ref_df = pd.DataFrame(team_ref_rows)
    result['REF_MIN_GGR_PLAYER_TEAM'] = pd.to_numeric(_team_ref_df['REF_MIN_GGR_PLAYER_TEAM'], errors='coerce').values
    result['REF_MAX_GGR_EUR_TEAM']    = pd.to_numeric(_team_ref_df['REF_MAX_GGR_EUR_TEAM'], errors='coerce').values
    result['REF_MIN_GGR_EUR_TEAM']    = pd.to_numeric(_team_ref_df['REF_MIN_GGR_EUR_TEAM'], errors='coerce').values
except NameError:
    result['REF_MIN_GGR_PLAYER_TEAM'] = np.nan
    result['REF_MAX_GGR_EUR_TEAM'] = np.nan
    result['REF_MIN_GGR_EUR_TEAM'] = np.nan

result['REF_MIN_GGR_PLAYER_TEAM'] = pd.to_numeric(result['REF_MIN_GGR_PLAYER_TEAM'], errors='coerce')
result['REF_MAX_GGR_EUR_TEAM']    = pd.to_numeric(result['REF_MAX_GGR_EUR_TEAM'], errors='coerce')
result['REF_MIN_GGR_EUR_TEAM']    = pd.to_numeric(result['REF_MIN_GGR_EUR_TEAM'], errors='coerce')

# --- PLAYER-based ratio (AMENDED): weakest side's |cumulative PLAYER-GGR| vs
#     that team's |REF_MIN_GGR_PLAYER_TEAM| at the band ---
def _get_cum_ggr_player_lowest(r):
    pfx, low = r.get('LAST_SNAPSHOT'), r.get('LOWEST_GGR_STAKE_OUTCOME')
    return r.get(f'{pfx}_CUM_GGR_PLAYER_{low}') if pfx and low else None

result['_CUM_GGR_PLAYER_LOWEST'] = pd.to_numeric(
    result.apply(_get_cum_ggr_player_lowest, axis=1), errors='coerce'
)

result['RATIO_GGR_PLAYER_TO_TEAM_MAX'] = np.where(
    result['REF_MIN_GGR_PLAYER_TEAM'].notna()
        & (result['REF_MIN_GGR_PLAYER_TEAM'].abs() > 0)
        & result['_CUM_GGR_PLAYER_LOWEST'].notna(),
    (result['_CUM_GGR_PLAYER_LOWEST'].abs() / result['REF_MIN_GGR_PLAYER_TEAM'].abs()), np.nan)
result['RATIO_GGR_PLAYER_TO_TEAM_MAX'] = result['RATIO_GGR_PLAYER_TO_TEAM_MAX'].astype(float).round(4)

# Adjusted probability — kept multiplying the headline predicted-outcome
# probability, as before. (If you want the player track to mirror the stake
# track fully, swap M_LAST_SNAPSHOT_PRED_PROB for
# M_LAST_SNAPSHOT_LOWEST_GGR_STAKE_PROB below.)
result['M_LAST_SNAPSHOT_PRED_PROB_ADJUSTED_TEAM'] = np.where(
    result['M_LAST_SNAPSHOT_PRED_PROB'].notna() & result['RATIO_GGR_PLAYER_TO_TEAM_MAX'].notna(),
    (result['RATIO_GGR_PLAYER_TO_TEAM_MAX'] * result['M_LAST_SNAPSHOT_PRED_PROB']), np.nan)
result['M_LAST_SNAPSHOT_PRED_PROB_ADJUSTED_TEAM'] = result['M_LAST_SNAPSHOT_PRED_PROB_ADJUSTED_TEAM'].astype(float).round(4)

# --- STAKE-based ratio (unchanged): outcome with the LOWEST current cumulative
#     estimated GGR_STAKE vs. that weakest side's team historical MIN cumulative
#     GGR_STAKE at the band (REF_MIN_GGR_EUR_TEAM, built above). ---
def _get_cum_ggr_stake_lowest(r):
    pfx, low = r.get('LAST_SNAPSHOT'), r.get('LOWEST_GGR_STAKE_OUTCOME')
    return r.get(f'{pfx}_CUM_GGR_STAKE_{low}') if pfx and low else None

result['_CUM_GGR_STAKE_LOWEST'] = pd.to_numeric(
    result.apply(_get_cum_ggr_stake_lowest, axis=1), errors='coerce'
)

result['RATIO_GGR_STAKE_TO_TEAM_MAX'] = np.where(
    result['REF_MIN_GGR_EUR_TEAM'].notna() & (result['REF_MIN_GGR_EUR_TEAM'].abs() > 0) & result['_CUM_GGR_STAKE_LOWEST'].notna(),
    (result['_CUM_GGR_STAKE_LOWEST'].abs() / result['REF_MIN_GGR_EUR_TEAM'].abs()), np.nan)
result['RATIO_GGR_STAKE_TO_TEAM_MAX'] = result['RATIO_GGR_STAKE_TO_TEAM_MAX'].astype(float).round(4)

def _lowest_ggr_stake_prob(r):
    pfx, low = r.get('LAST_SNAPSHOT'), r.get('LOWEST_GGR_STAKE_OUTCOME')
    return r.get(f'{pfx}_PROB_{low}') if pfx and low else None

result['M_LAST_SNAPSHOT_LOWEST_GGR_STAKE_PROB'] = pd.to_numeric(
    result.apply(_lowest_ggr_stake_prob, axis=1), errors='coerce'
)
result['M_LAST_SNAPSHOT_PRED_PROB_ADJUSTED_TEAM_STAKE'] = np.where(
    result['M_LAST_SNAPSHOT_LOWEST_GGR_STAKE_PROB'].notna() & result['RATIO_GGR_STAKE_TO_TEAM_MAX'].notna(),
    (result['RATIO_GGR_STAKE_TO_TEAM_MAX'] * result['M_LAST_SNAPSHOT_LOWEST_GGR_STAKE_PROB']), np.nan)
result['M_LAST_SNAPSHOT_PRED_PROB_ADJUSTED_TEAM_STAKE'] = result['M_LAST_SNAPSHOT_PRED_PROB_ADJUSTED_TEAM_STAKE'].astype(float).round(4)

# Defensive drop: only remove temp columns that actually exist, so the cell
# can be re-run standalone (STEP 9/9b create the *_LAST columns; a previous
# run of this cell may already have dropped them).
_tmp_cols = ['_CUM_GGR_PLAYER_LAST', '_CUM_GGR_STAKE_LAST',
             '_CUM_N_PLAYERS_LAST', '_CUM_GGR_STAKE_LOWEST',
             '_CUM_GGR_PLAYER_LOWEST']
result.drop(columns=[c for c in _tmp_cols if c in result.columns], inplace=True)
# ─── STEP 9f: Fundamentals model join v2 (PROD.SBX_BI.DD_FOOTBALL_FUNDAMENT_MODEL) ─
# v2 JOIN: the v1 exact/fuzzy(0.85) join missed in-scope events whose feed
# names carry prefixes/suffixes football-data drops ("AC Milan" vs "Milan",
# "Real Sociedad" vs "Sociedad", "PAOK Thessaloniki" vs "PAOK"). New pipeline:
#   0. LEAGUE SCOPING — TOURNAMENT_NAME is mapped to the fundamentals league
#      key (country + competition keywords, e.g. "Italy - Serie A" → serie_a,
#      NOT Brazil's Serie A), so candidates shrink to ~20 teams.
#   1. EXACT on normalised full name
#   2. ALIAS dictionary (seeded with known feed ↔ football-data renames)
#   3. TOKEN match — generic tokens (fc/ac/real/...) stripped; a side matches
#      when every remaining token of the shorter name is covered by the other
#      (token equal, or prefix ≥3 chars: "nottm"~"nottingham")
#   4. difflib fuzzy fallback: cutoff 0.75 league-scoped, 0.85 global
# FUND_MATCH_TYPE records the weakest stage used across the two sides
# (EXACT/ALIAS/TOKEN/FUZZY/NO_MATCH); FUND_TEAM1/2_MATCH keep the matched
# football-data names for audit.
# Divergence per outcome:  FUND_DIVERGENCE_o = CUM_P_o(last snapshot) − FUND_P_o
#   > 0 → betting FLOW prices the outcome ABOVE fundamentals (alert direction).
# NOTE: fundamentals are PRE-MATCH — treat divergence as an alerting signal
# pre-KO..~M30; later bands absorb the live score and diverge naturally.
FUND_FUZZY_LEAGUE_CUTOFF = 0.75
FUND_FUZZY_GLOBAL_CUTOFF = 0.85

def _fund_ascii(s):
    return unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode().lower()

def _norm_team_fund(s):
    return re.sub(r'[^a-z0-9]', '', _fund_ascii(s))

_FUND_TOKEN_STOP = {'fc', 'cf', 'ac', 'sc', 'afc', 'cfc', 'ca', 'cd', 'cp', 'sk', 'fk',
                    'ks', 'bk', 'if', 'us', 'uc', 'ss', 'ssc', 'club', 'de', 'the',
                    'real', 'sporting', 'deportivo', 'athletic', 'atletico', 'olympique',
                    'sv', 'vfb', 'vfl', 'fsv', 'tsg', 'rc', 'og', 'as', '1', 'i', 'ii'}

def _fund_tokens(s):
    toks = [t for t in re.sub(r'[^a-z0-9 ]', ' ', _fund_ascii(s)).split() if t]
    core = [t for t in toks if t not in _FUND_TOKEN_STOP]
    return core if core else toks          # never strip a name to nothing

def _tok_eq(a, b):
    return a == b or (len(a) >= 3 and b.startswith(a)) or (len(b) >= 3 and a.startswith(b))

def _token_cover(q_toks, c_toks):
    """True if every token of the SHORTER list is covered by the other side."""
    short, long_ = (q_toks, c_toks) if len(q_toks) <= len(c_toks) else (c_toks, q_toks)
    return len(short) > 0 and all(any(_tok_eq(t, u) for u in long_) for t in short)

# feed-name → football-data-name aliases (normalised keys/values); extend as
# FUND_MATCH_TYPE audits surface new misses.
_FUND_ALIASES = {
    'acmilan': 'milan', 'internazionale': 'inter', 'intermilan': 'inter',
    'realsociedad': 'sociedad', 'athleticbilbao': 'athbilbao',
    'atleticomadrid': 'athmadrid', 'espanyol': 'espanol', 'realbetis': 'betis',
    'celtavigo': 'celta', 'manchestercity': 'mancity', 'manchesterunited': 'manunited',
    'nottinghamforest': 'nottmforest', 'sheffieldunited': 'sheffieldutd',
    'wolverhampton': 'wolves', 'wolverhamptonwanderers': 'wolves',
    'psg': 'parissg', 'parissaintgermain': 'parissg', 'olympiquemarseille': 'marseille',
    'olympiquelyonnais': 'lyon', 'saintetienne': 'stetienne',
    'bayernmunchen': 'bayernmunich', 'borussiamonchengladbach': 'mgladbach',
    'monchengladbach': 'mgladbach', 'koln': 'fckoln', '1fckoln': 'fckoln',
    'eintrachtfrankfurt': 'einfrankfurt', 'bayerleverkusen': 'leverkusen',
    'borussiadortmund': 'dortmund', 'sportingcp': 'splisbon', 'sportinglisbon': 'splisbon',
    'vitoriaguimaraes': 'guimaraes', 'paokthessaloniki': 'paok',
    'olympiakospiraeus': 'olympiakos', 'aekathens': 'aek',
    'panathinaikosathens': 'panathinaikos', 'fcsb': 'fcsb', 'steauabucuresti': 'fcsb',
    'universitateacraiova': 'ucraiova', 'cfrcluj': 'cfrcluj',
}

# TOURNAMENT_NAME → fundamentals league key (country keyword + competition
# keyword, checked in order; LaLiga 2 / 2. Bundesliga / Serie B never map).
_FUND_LEAGUE_RULES = [
    (('england',), ('premier league',), 'premier_league'),
    (('england',), ('championship',), 'championship'),
    (('spain',), ('laliga 2', 'laliga2', 'segunda'), None),
    (('spain',), ('laliga', 'la liga', 'primera'), 'la_liga'),
    (('germany',), ('2. bundesliga', 'bundesliga 2', '2 bundesliga'), 'bundesliga_2'),
    (('germany',), ('bundesliga',), 'bundesliga'),
    (('italy',), ('serie b',), None),
    (('italy',), ('serie a',), 'serie_a'),
    (('france',), ('ligue 2',), None),
    (('france',), ('ligue 1',), 'ligue_1'),
    (('belgium',), ('pro league', 'first division', 'jupiler'), 'belgium'),
    (('greece',), ('super league',), 'greek'),
    (('romania',), ('superliga', 'liga 1', 'liga i'), 'romania'),
    (('poland',), ('ekstraklasa',), 'ekstraklasa'),
    (('netherlands',), ('eredivisie',), 'eredivisie'),
    (('portugal',), ('primeira', 'liga portugal'), 'portugal'),
    (('turkey', 'turkiye'), ('super lig', 'süper lig', 'superlig'), 'turkey'),
    (('scotland',), ('premiership',), 'scotland'),
]

def _fund_league_for(tournament_name):
    t = _fund_ascii(tournament_name)
    for _countries, _comps, _key in _FUND_LEAGUE_RULES:
        if any(c in t for c in _countries) and any(k in t for k in _comps):
            return _key
    return None

_fund_cols_out = ['FUND_P_1', 'FUND_P_X', 'FUND_P_2', 'FUND_PRED', 'FUND_CONFIDENCE',
                  'FUND_LEAGUE', 'FUND_MATCH_TYPE', 'FUND_TEAM1_MATCH', 'FUND_TEAM2_MATCH',
                  'FUND_DIVERGENCE_1', 'FUND_DIVERGENCE_X', 'FUND_DIVERGENCE_2',
                  'FUND_DIVERGENCE_PRED']
try:
    from snowflake.snowpark.context import get_active_session
    _fund = get_active_session().sql(
        'SELECT LEAGUE, TEAM1, TEAM2, TEAM1_NORM, TEAM2_NORM, '
        '       FUND_P_1, FUND_P_X, FUND_P_2, FUND_PRED, FUND_CONFIDENCE '
        'FROM PROD.SBX_BI.DD_FOOTBALL_FUNDAMENT_MODEL'
    ).to_pandas()
    _fund_pair = {(r.LEAGUE, r.TEAM1_NORM, r.TEAM2_NORM): r for r in _fund.itertuples(index=False)}
    # per-league and global candidate registries: norm -> original fd name
    _fund_teams_by_lg = {}
    for _r in _fund.itertuples(index=False):
        _fund_teams_by_lg.setdefault(_r.LEAGUE, {})[_r.TEAM1_NORM] = _r.TEAM1
        _fund_teams_by_lg.setdefault(_r.LEAGUE, {})[_r.TEAM2_NORM] = _r.TEAM2
    _fund_teams_global = {}
    for _d in _fund_teams_by_lg.values():
        _fund_teams_global.update(_d)
    _fund_toks_cache = {n: _fund_tokens(orig) for _d in _fund_teams_by_lg.values()
                        for n, orig in _d.items()}

    _STAGE_RANK = {'EXACT': 0, 'ALIAS': 1, 'TOKEN': 2, 'FUZZY': 3, 'NO_MATCH': 9}

    def _fund_resolve(name, lg_key):
        """→ (fd_norm, fd_name, stage). League-scoped candidates when lg_key
        is known, global otherwise."""
        cands = _fund_teams_by_lg.get(lg_key) if lg_key else _fund_teams_global
        if not cands:
            cands = _fund_teams_global
        n = _norm_team_fund(name)
        if n in cands:
            return n, cands[n], 'EXACT'
        a = _FUND_ALIASES.get(n)
        if a and a in cands:
            return a, cands[a], 'ALIAS'
        q_toks = _fund_tokens(name)
        tok_hits = [c for c in cands if _token_cover(q_toks, _fund_toks_cache.get(c, []))]
        if len(tok_hits) == 1:
            return tok_hits[0], cands[tok_hits[0]], 'TOKEN'
        _cut = FUND_FUZZY_LEAGUE_CUTOFF if lg_key else FUND_FUZZY_GLOBAL_CUTOFF
        close = difflib.get_close_matches(n, list(cands), n=2, cutoff=_cut)
        if close and (len(close) == 1 or
                      difflib.SequenceMatcher(None, n, close[0]).ratio()
                      - difflib.SequenceMatcher(None, n, close[1]).ratio() > 0.05):
            return close[0], cands[close[0]], 'FUZZY'
        return None, None, 'NO_MATCH'

    _rows9f = []
    for idx, r in result.iterrows():
        rec = {c: None for c in _fund_cols_out}
        if r.get('SPORT_NAME') == 'Soccer':
            t1, t2 = _split_teams(r.get('EVENT_NAME'))
            if t1 and t2:
                _lg_key = _fund_league_for(r.get('TOURNAMENT_NAME'))
                n1, f1, m1 = _fund_resolve(t1, _lg_key)
                n2, f2, m2 = _fund_resolve(t2, _lg_key)
                hit = None
                if n1 and n2:
                    if _lg_key:
                        hit = _fund_pair.get((_lg_key, n1, n2))
                    if hit is None:      # both resolved globally → find their league
                        for _l in _fund_teams_by_lg:
                            hit = _fund_pair.get((_l, n1, n2))
                            if hit is not None:
                                break
                if hit is not None:
                    rec['FUND_P_1'] = float(hit.FUND_P_1)
                    rec['FUND_P_X'] = float(hit.FUND_P_X)
                    rec['FUND_P_2'] = float(hit.FUND_P_2)
                    rec['FUND_PRED'] = str(hit.FUND_PRED)
                    rec['FUND_CONFIDENCE'] = float(hit.FUND_CONFIDENCE)
                    rec['FUND_LEAGUE'] = str(hit.LEAGUE)
                    rec['FUND_TEAM1_MATCH'], rec['FUND_TEAM2_MATCH'] = f1, f2
                    rec['FUND_MATCH_TYPE'] = max(m1, m2, key=lambda s: _STAGE_RANK[s])
                    for _o in ['1', 'X', '2']:
                        _cp = r.get(f'M_LAST_SNAPSHOT_PROB_{_o}')
                        rec[f'FUND_DIVERGENCE_{_o}'] = (round(float(_cp) - rec[f'FUND_P_{_o}'], 4)
                                                        if _cp is not None and not pd.isna(_cp) else None)
                    _pred = r.get('M_LAST_SNAPSHOT_PRED')
                    rec['FUND_DIVERGENCE_PRED'] = rec.get(f'FUND_DIVERGENCE_{_pred}') if _pred else None
                else:
                    rec['FUND_MATCH_TYPE'] = 'NO_MATCH' if _lg_key else 'OUT_OF_SCOPE'
        _rows9f.append(rec)
    _f9 = pd.DataFrame(_rows9f)
    for _c in _fund_cols_out:
        result[_c] = _f9[_c].values
    _n_soc = int((result['SPORT_NAME'] == 'Soccer').sum())
    _mt = _f9['FUND_MATCH_TYPE'].value_counts(dropna=True).to_dict()
    print(f'STEP 9f: fundamentals join over {_n_soc} soccer events — {_mt}')
    _miss = result.loc[result['FUND_MATCH_TYPE'] == 'NO_MATCH', 'EVENT_NAME'].tolist()
    if _miss:
        print(f'STEP 9f: in-scope leagues but unmatched (add aliases?): {_miss}')
except Exception as _e9f:
    print(f'STEP 9f: fundamentals table unavailable ({_e9f}) — FUND_* columns set to None')
    for _c in _fund_cols_out:
        result[_c] = None
# ─── STEP 9e: Team squad value — Top 11 players by CURRENT_VALUE per team ───
try:
    _val = dataframe_2.copy() if isinstance(dataframe_2, pd.DataFrame) else dataframe_2.to_pandas()
    _val.columns = [c.upper() for c in _val.columns]
    _val['VALUE_EUR'] = _val['CURRENT_VALUE'].apply(_parse_value_eur)
    _val['COMPETITION_NORM'] = _val['COMPETITION'].apply(_normalize_name)

    # Top-11 value per (COMPETITION, CLUB) — keeps clubs scoped to their competition
    team_top11 = (
        _val.groupby(['COMPETITION', 'CLUB'])['VALUE_EUR']
            .apply(lambda s: s.dropna().nlargest(11).sum())
            .rename('TOP11_VALUE_EUR')
            .reset_index()
    )
    team_top11['CLUB_NORM'] = team_top11['CLUB'].apply(_normalize_name)
    team_top11['COMPETITION_NORM'] = team_top11['COMPETITION'].apply(_normalize_name)

    # Global (competition-agnostic) index — fallback path
    team_top11_global = (
        team_top11.groupby('CLUB')['TOP11_VALUE_EUR']
            .max()
            .reset_index()
    )
    team_top11_global['CLUB_NORM'] = team_top11_global['CLUB'].apply(_normalize_name)
    _global_map = dict(zip(team_top11_global['CLUB_NORM'], team_top11_global['TOP11_VALUE_EUR']))
    _global_norm_list = team_top11_global['CLUB_NORM'].tolist()
    _global_norm_to_original = dict(zip(team_top11_global['CLUB_NORM'], team_top11_global['CLUB']))

    _comp_norm_list = team_top11['COMPETITION_NORM'].unique().tolist()
    _comp_norm_to_original = dict(zip(team_top11['COMPETITION_NORM'], team_top11['COMPETITION']))

    def _match_competition(tournament_name):
        full_norm, league_norm = _normalize_competition(tournament_name)
        for candidate in (league_norm, full_norm):
            if not candidate:
                continue
            if candidate in _comp_norm_to_original:
                return _comp_norm_to_original[candidate], 'EXACT', 1.0
            alias = COMPETITION_NAME_ALIASES.get(candidate)
            if alias and alias in _comp_norm_to_original:
                return _comp_norm_to_original[alias], 'ALIAS', 1.0
        for candidate in (league_norm, full_norm):
            if not candidate:
                continue
            close = difflib.get_close_matches(candidate, _comp_norm_list, n=1, cutoff=COMPETITION_FUZZY_CUTOFF)
            if close:
                score = difflib.SequenceMatcher(None, candidate, close[0]).ratio()
                return _comp_norm_to_original[close[0]], 'FUZZY', round(score, 3)
        return None, 'NO_MATCH', None

    def _match_team_in_competition(team_name, competition_original):
        if team_name is None or competition_original is None:
            return None, None, 'NONE', None
        subset = team_top11[team_top11['COMPETITION'] == competition_original]
        if subset.empty:
            return None, None, 'NO_MATCH', None
        local_map = dict(zip(subset['CLUB_NORM'], subset['TOP11_VALUE_EUR']))
        local_norm_list = subset['CLUB_NORM'].tolist()
        local_norm_to_original = dict(zip(subset['CLUB_NORM'], subset['CLUB']))
        norm = _normalize_name(team_name)
        if norm in local_map:
            return local_norm_to_original[norm], local_map[norm], 'EXACT', 1.0
        alias = TEAM_NAME_ALIASES.get(norm)
        if alias and alias in local_map:
            return local_norm_to_original[alias], local_map[alias], 'ALIAS', 1.0
        close = difflib.get_close_matches(norm, local_norm_list, n=1, cutoff=FUZZY_CUTOFF)
        if close:
            score = difflib.SequenceMatcher(None, norm, close[0]).ratio()
            return local_norm_to_original[close[0]], local_map[close[0]], 'FUZZY', round(score, 3)
        return None, None, 'NO_MATCH', None

    def _match_team_global(team_name):
        if team_name is None:
            return None, None, 'NONE', None
        norm = _normalize_name(team_name)
        if norm in _global_map:
            return _global_norm_to_original[norm], _global_map[norm], 'GLOBAL_EXACT', 1.0
        alias = TEAM_NAME_ALIASES.get(norm)
        if alias and alias in _global_map:
            return _global_norm_to_original[alias], _global_map[alias], 'GLOBAL_ALIAS', 1.0
        close = difflib.get_close_matches(norm, _global_norm_list, n=1, cutoff=FUZZY_CUTOFF)
        if close:
            score = difflib.SequenceMatcher(None, norm, close[0]).ratio()
            return _global_norm_to_original[close[0]], _global_map[close[0]], 'GLOBAL_FUZZY', round(score, 3)
        return None, None, 'NO_MATCH', None

    def _match_team(team_name, competition_original):
        club, val, method, score = _match_team_in_competition(team_name, competition_original)
        if method in ('EXACT', 'ALIAS', 'FUZZY'):
            return club, val, method, score
        return _match_team_global(team_name)

    comp_match_name, comp_match_type, comp_match_score = [], [], []
    t1_val, t2_val = [], []
    t1_club, t2_club = [], []
    t1_method, t2_method = [], []
    t1_score, t2_score = [], []
    for idx, r in result.iterrows():
        if not is_soccer.loc[idx]:
            comp_match_name.append(None); comp_match_type.append('NOT_SOCCER'); comp_match_score.append(None)
            t1_val.append(None); t2_val.append(None)
            t1_club.append(None); t2_club.append(None)
            t1_method.append('NOT_SOCCER'); t2_method.append('NOT_SOCCER')
            t1_score.append(None); t2_score.append(None)
            continue
        comp_name, comp_method, comp_score = _match_competition(r.get('TOURNAMENT_NAME'))
        comp_match_name.append(comp_name); comp_match_type.append(comp_method); comp_match_score.append(comp_score)

        t1, t2 = _split_teams(r.get('EVENT_NAME'))
        c1, v1, m1, s1 = _match_team(t1, comp_name)
        c2, v2, m2, s2 = _match_team(t2, comp_name)
        t1_club.append(c1); t1_val.append(v1); t1_method.append(m1); t1_score.append(s1)
        t2_club.append(c2); t2_val.append(v2); t2_method.append(m2); t2_score.append(s2)

    result['TOURNAMENT_VALUE_MATCH_NAME']  = comp_match_name
    result['TOURNAMENT_VALUE_MATCH_TYPE']  = comp_match_type
    result['TOURNAMENT_VALUE_MATCH_SCORE'] = comp_match_score
    result['TEAM1_TOP11_VALUE_EUR']   = t1_val
    result['TEAM1_VALUE_MATCH_NAME']  = t1_club
    result['TEAM1_VALUE_MATCH_TYPE']  = t1_method
    result['TEAM1_VALUE_MATCH_SCORE'] = t1_score
    result['TEAM2_TOP11_VALUE_EUR']   = t2_val
    result['TEAM2_VALUE_MATCH_NAME']  = t2_club
    result['TEAM2_VALUE_MATCH_TYPE']  = t2_method
    result['TEAM2_VALUE_MATCH_SCORE'] = t2_score
    result['TOP11_VALUE_DIFF_EUR'] = np.where(
        is_soccer,
        (pd.to_numeric(result['TEAM1_TOP11_VALUE_EUR'], errors='coerce')
         - pd.to_numeric(result['TEAM2_TOP11_VALUE_EUR'], errors='coerce')).round(2),
        np.nan
    )
except NameError:
    for _c in ['TOURNAMENT_VALUE_MATCH_NAME','TOURNAMENT_VALUE_MATCH_TYPE','TOURNAMENT_VALUE_MATCH_SCORE',
               'TEAM1_TOP11_VALUE_EUR','TEAM1_VALUE_MATCH_NAME','TEAM1_VALUE_MATCH_TYPE','TEAM1_VALUE_MATCH_SCORE',
               'TEAM2_TOP11_VALUE_EUR','TEAM2_VALUE_MATCH_NAME','TEAM2_VALUE_MATCH_TYPE','TEAM2_VALUE_MATCH_SCORE',
               'TOP11_VALUE_DIFF_EUR']:
        result[_c] = None
# ─── STEP 9d: Limit output — ranked by pre-game + early in-play player volume ─
PRE_KO_FALLBACK_PREFIXES  = ['KO', 'PRE20', 'PRE30', 'PRE40', 'PRE50', 'PRE60']
POST_KO_FALLBACK_PREFIXES = ['M30', 'M20', 'M10']

def _players_pre60(row):
    for pfx in PRE_KO_FALLBACK_PREFIXES:
        v = row.get(f'{pfx}_CUM_N_PLAYERS')
        if v is not None and not pd.isna(v):
            return float(v)
    return 0.0

def _players_first30(row):
    ko_val = row.get('KO_CUM_N_PLAYERS')
    ko_val = float(ko_val) if (ko_val is not None and not pd.isna(ko_val)) else None
    for pfx in POST_KO_FALLBACK_PREFIXES:
        v = row.get(f'{pfx}_CUM_N_PLAYERS')
        if v is not None and not pd.isna(v):
            v = float(v)
            return max(v - ko_val, 0.0) if ko_val is not None else v
    return 0.0

result['PLAYERS_PRE60_WINDOW']   = result.apply(_players_pre60, axis=1)
result['PLAYERS_FIRST30_WINDOW'] = result.apply(_players_first30, axis=1)
result['RANKING_METRIC'] = result['PLAYERS_PRE60_WINDOW'] + result['PLAYERS_FIRST30_WINDOW']

is_wc2026 = result['TOURNAMENT_NAME'].str.contains('World Cup 2026', case=False, na=False)
wc2026_rows = result.loc[is_wc2026]
non_wc_rows = result.loc[~is_wc2026]

PER_SPORT_CAPS = {'Soccer': 25,  'Basketball': 5}

capped_groups = []
for sport, cap in PER_SPORT_CAPS.items():
    grp = non_wc_rows.loc[non_wc_rows['SPORT_NAME'] == sport]
    capped_groups.append(grp.sort_values('RANKING_METRIC', ascending=False).head(cap))

other_sports_rows = non_wc_rows.loc[~non_wc_rows['SPORT_NAME'].isin(PER_SPORT_CAPS.keys())]

result = (
    pd.concat([wc2026_rows, *capped_groups, other_sports_rows], ignore_index=True)
      .sort_values('RANKING_METRIC', ascending=False)
      .reset_index(drop=True)
)
# ─── STEP 10: Final output ───────────────────────────────────────────────
OUTPUT_COLS = [
    'REAL_START_TIME_PLUS_1H','TOURNAMENT_NAME', 'EVENT_NAME', 'SPORT_NAME',
    'N_PLAYERS_KO_WINDOW_10MIN',
    'LAST_SNAPSHOT','M_LAST_SNAPSHOT_PRED','SIGNAL_CONSISTENCY_PCT',
    'M_LAST_SNAPSHOT_CUM_GGR_STAKE_1','M_LAST_SNAPSHOT_CUM_GGR_STAKE_X','M_LAST_SNAPSHOT_CUM_GGR_STAKE_2',
    'M_LAST_SNAPSHOT_CUM_GGR_PLAYER_1','M_LAST_SNAPSHOT_CUM_GGR_PLAYER_X','M_LAST_SNAPSHOT_CUM_GGR_PLAYER_2',
    # ── Post-kickoff aggregates: sum of per-band GGR across in-play bands only
    #    (BAND_ORDER >= 7, i.e. "0 to +9 min" onwards), per outcome.
    'CUM_GGR_STAKE_POSTKO_1','CUM_GGR_STAKE_POSTKO_X','CUM_GGR_STAKE_POSTKO_2',
    'CUM_GGR_PLAYER_POSTKO_1','CUM_GGR_PLAYER_POSTKO_X','CUM_GGR_PLAYER_POSTKO_2',
    # ── KO band = "−10 to −1 min" → the pre-kickoff band ending at event start.
    #    Per-band (non-cumulative) GGR generated in that window only.
    'KO_GGR_STAKE_1','KO_GGR_STAKE_X','KO_GGR_STAKE_2',
    'KO_GGR_PLAYER_1','KO_GGR_PLAYER_X','KO_GGR_PLAYER_2',
    'M10_GGR_STAKE_1','M10_GGR_STAKE_X','M10_GGR_STAKE_2',
    'M10_GGR_PLAYER_1','M10_GGR_PLAYER_X','M10_GGR_PLAYER_2',
    'AVG_ODDS_PRE_1','AVG_ODDS_PRE_X','AVG_ODDS_PRE_2',
    #'LAST_ODDS_1','LAST_ODDS_X','LAST_ODDS_2',
    # ── Odds drift (NEW): DELTA_ODDS_o = LAST_ODDS_o / AVG_ODDS_PRE_o
    'DELTA_ODDS_1','DELTA_ODDS_X','DELTA_ODDS_2',
    'KO_CUM_GGR_STAKE_1','KO_CUM_GGR_STAKE_X','KO_CUM_GGR_STAKE_2',
    'KO_CUM_GGR_PLAYER_1','KO_CUM_GGR_PLAYER_X','KO_CUM_GGR_PLAYER_2',
    'M20_GGR_STAKE_1','M20_GGR_STAKE_X','M20_GGR_STAKE_2',
    'M20_GGR_PLAYER_1','M20_GGR_PLAYER_X','M20_GGR_PLAYER_2',
    'M30_GGR_STAKE_1','M30_GGR_STAKE_X','M30_GGR_STAKE_2',
    'M30_GGR_PLAYER_1','M30_GGR_PLAYER_X','M30_GGR_PLAYER_2',
    'M40_GGR_STAKE_1','M40_GGR_STAKE_X','M40_GGR_STAKE_2',
    'M40_GGR_PLAYER_1','M40_GGR_PLAYER_X','M40_GGR_PLAYER_2',
    'M50_GGR_STAKE_1','M50_GGR_STAKE_X','M50_GGR_STAKE_2',
    'M50_GGR_PLAYER_1','M50_GGR_PLAYER_X','M50_GGR_PLAYER_2',
    'M60_GGR_STAKE_1','M60_GGR_STAKE_X','M60_GGR_STAKE_2',
    'M60_GGR_PLAYER_1','M60_GGR_PLAYER_X','M60_GGR_PLAYER_2',
    'M70_GGR_STAKE_1','M70_GGR_STAKE_X','M70_GGR_STAKE_2',
    'M70_GGR_PLAYER_1','M70_GGR_PLAYER_X','M70_GGR_PLAYER_2',
     'M80_GGR_STAKE_1','M80_GGR_STAKE_X','M80_GGR_STAKE_2',
    'M80_GGR_PLAYER_1','M80_GGR_PLAYER_X','M80_GGR_PLAYER_2',
    'M90_GGR_STAKE_1','M90_GGR_STAKE_X','M90_GGR_STAKE_2',
    'M90_GGR_PLAYER_1','M90_GGR_PLAYER_X','M90_GGR_PLAYER_2',
    'M100_GGR_STAKE_1','M100_GGR_STAKE_X','M100_GGR_STAKE_2',
    'M100_GGR_PLAYER_1','M100_GGR_PLAYER_X','M100_GGR_PLAYER_2',
    'CONF_DELTA_KO_TO_M_LAST_SNAPSHOT','PRED_STABILITY','SIGNAL_TREND',
    'M_LAST_SNAPSHOT_PRED_PROB',
    'M_LAST_SNAPSHOT_PROB_STAKE','M_LAST_SNAPSHOT_PROB_PLAYER',
    # ── Fundamentals model (STEP 9f): pre-match P(1/X/2) + flow-vs-fundamentals
    #    divergence at the last snapshot, per outcome and for the predicted one.
    'FUND_P_1','FUND_P_X','FUND_P_2','FUND_PRED','FUND_CONFIDENCE',
    'FUND_LEAGUE','FUND_MATCH_TYPE','FUND_TEAM1_MATCH','FUND_TEAM2_MATCH',
    'FUND_DIVERGENCE_1','FUND_DIVERGENCE_X','FUND_DIVERGENCE_2',
    'FUND_DIVERGENCE_PRED',
    'RATIO_GGR_PLAYER_TO_TOURNAMENT_MAX',
    'RATIO_GGR_STAKE_TO_TOURNAMENT_MAX',
    'REF_MIN_GGR_PLAYER_TEAM','RATIO_GGR_PLAYER_TO_TEAM_MAX',
    'LOWEST_GGR_STAKE_OUTCOME','M_LAST_SNAPSHOT_LOWEST_GGR_STAKE_PROB',
    'REF_MIN_GGR_EUR_TEAM','RATIO_GGR_STAKE_TO_TEAM_MAX',

]
result = result[[c for c in OUTPUT_COLS if c in result.columns]]
odds_cols = ['AVG_ODDS_PRE_1','AVG_ODDS_PRE_X','AVG_ODDS_PRE_2','LAST_ODDS_1','LAST_ODDS_X','LAST_ODDS_2']
is_wc2026_final = result['TOURNAMENT_NAME'].str.contains('World Cup 2026', case=False, na=False)
wc2026_final = result.loc[is_wc2026_final]
other_final = result.loc[~is_wc2026_final].dropna(subset=[c for c in odds_cols if c in result.columns], how='all')
result = pd.concat([wc2026_final, other_final], ignore_index=True)

# ─── STEP 11: Tournament GGR & Margin — last activity day + current day ──────
# Source: "League results" cell (dataframe_3) = per-event daily history from
# PROD.SBX_BI.DD_HISTORY_LONG_TERM over the last 10 days.
# Per tournament (World-Cup-normalised), GGR and STAKE are summed per
# EVENT_DATE, then combined over the two relevant days:
#     TOURNAMENT_GGR_LAST_DAY_PLUS_TODAY    = GGR(last activity day before
#                                             today) + GGR(today)
#     TOURNAMENT_Margin_LAST_DAY_PLUS_TODAY = pooled margin % over the same
#                                             two days:
#                                             (GGR_last + GGR_today)
#                                             / (STAKE_last + STAKE_today) * 100
#     (stake-weighted pool, NOT an average of two daily margins, so a small
#      yesterday can't distort a big today; zero combined stake -> None)
# Joined to `result` via TOURNAMENT_NAME and appended AFTER the OUTPUT_COLS
# slice so they are always the LAST two columns. A tournament absent from the
# 10-day window gives NaN in both columns.
try:
    _lg = dataframe_3.copy() if isinstance(dataframe_3, pd.DataFrame) else dataframe_3.to_pandas()
    _lg.columns = [c.upper() for c in _lg.columns]
    _lg['TOURNAMENT_NAME'] = _lg['TOURNAMENT_NAME'].fillna('UNKNOWN')
    _lg['TOURNAMENT_NAME'] = _lg['TOURNAMENT_NAME'].where(
        ~_lg['TOURNAMENT_NAME'].str.contains('World Cup 2026', case=False, na=False),
        other='World Cup 2026')
    _lg['EVENT_DATE'] = pd.to_datetime(_lg['EVENT_DATE'], errors='coerce').dt.normalize()
    _ggr_col   = 'GGR' if 'GGR' in _lg.columns else 'GGR_EUR'
    _stake_col = 'STAKE_EUR' if 'STAKE_EUR' in _lg.columns else 'STAKES_EUR'
    _lg[_ggr_col]   = pd.to_numeric(_lg[_ggr_col],   errors='coerce').fillna(0.0)
    _lg[_stake_col] = pd.to_numeric(_lg[_stake_col], errors='coerce').fillna(0.0)
    _lg = _lg.dropna(subset=['EVENT_DATE'])

    _daily = _lg.groupby(['TOURNAMENT_NAME', 'EVENT_DATE'], as_index=False)[[_ggr_col, _stake_col]].sum()
    _today = pd.Timestamp.now().normalize()

    _ggr_map, _margin_map = {}, {}
    for _tourn, _g in _daily.groupby('TOURNAMENT_NAME'):
        _t = _g.loc[_g['EVENT_DATE'] == _today]
        _today_ggr   = float(_t[_ggr_col].sum())
        _today_stake = float(_t[_stake_col].sum())
        _prev = _g.loc[_g['EVENT_DATE'] < _today]
        if len(_prev) > 0:
            _last_day   = _prev['EVENT_DATE'].max()
            _l = _prev.loc[_prev['EVENT_DATE'] == _last_day]
            _last_ggr   = float(_l[_ggr_col].sum())
            _last_stake = float(_l[_stake_col].sum())
        else:
            _last_ggr, _last_stake = 0.0, 0.0
        _sum_ggr   = _last_ggr + _today_ggr
        _sum_stake = _last_stake + _today_stake
        # DD_HISTORY_LONG_TERM currently holds every row TWICE (uniform 2x
        # duplication: margin unaffected, absolute sums doubled). Correct the
        # GGR column by the duplication factor; margin needs no correction
        # since the factor cancels in the ratio. Set back to 1 once the
        # League results SQL dedupes (SELECT DISTINCT *) or the table is fixed.
        _GGR_DUP_FACTOR = 2
        _ggr_map[_tourn]    = round(_sum_ggr / _GGR_DUP_FACTOR, 2)
        _margin_map[_tourn] = round(_sum_ggr / _sum_stake * 100, 2) if _sum_stake > 0 else None

    _res_tourn_norm = result['TOURNAMENT_NAME'].fillna('UNKNOWN')
    _res_tourn_norm = _res_tourn_norm.where(
        ~_res_tourn_norm.str.contains('World Cup 2026', case=False, na=False),
        other='World Cup 2026')
    result['TOURNAMENT_GGR_LAST_DAY_PLUS_TODAY']    = _res_tourn_norm.map(_ggr_map)
    result['TOURNAMENT_Margin_LAST_DAY_PLUS_TODAY'] = _res_tourn_norm.map(_margin_map)
except NameError:
    # "League results" cell not run yet -> keep the columns, but empty
    result['TOURNAMENT_GGR_LAST_DAY_PLUS_TODAY']    = None
    result['TOURNAMENT_Margin_LAST_DAY_PLUS_TODAY'] = None

# ─── Format value columns to Millions (e.g. 22550000 -> "22.6 M") ────────────
def _fmt_millions(v):
    if pd.isna(v) or v is None:
        return None
    try:
        num = float(v)
    except (ValueError, TypeError):
        return None
    return f"{num / 1_000_000:.1f} M"

for _vc in ['TEAM1_TOP11_VALUE_EUR', 'TEAM2_TOP11_VALUE_EUR', 'TOP11_VALUE_DIFF_EUR']:
    if _vc in result.columns:
        result[_vc] = result[_vc].apply(_fmt_millions)
# ─── Display: local render only (skips the Snowflake temp-view upload that
#     the bare `result` expression triggers via df_to_snowflake_bundle) ──────
from IPython.display import display
display(result.head(50))

In [ ]:
import pandas as pd
import numpy as np
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# ── Memory-optimised pull (same treatment as the team-history cell) ──────────
# The previous version fetched RAW rows (every native-grain row, 30 days, all
# active tournaments) and did banding + cumsums in pandas via a per-row
# .apply(pd.Series) → kernel OOM / websocket disconnect.
# This version pushes EVERYTHING into SQL — banding (FLOOR-based, identical
# 16-band scheme), World Cup 2026 name normalisation, per-event cumulatives
# (window functions), and the tournament-level MIN/MAX aggregation — and
# returns only the FINAL frame: one row per TOURNAMENT × BAND × OUTCOME
# (a few hundred rows instead of millions).
#
# NOTE — one deliberate semantic change, aligning with the team cell:
# MIN/MAX_GGR_EUR_STAKE are now extremes over EVENT × OUTCOME × BAND *sums*,
# not over raw sub-band rows (the old min/max of arbitrary intra-band slices).
QUERY = """
WITH tournament AS (
    SELECT DISTINCT dt.TOURNAMENT_NAME,
           ds.SPORT_NAME
    FROM PROD.DWH.D_EVENT de
        JOIN PROD.DWH.D_TOURNAMENT dt
            ON dt.TOURNAMENT_ID = de.TOURNAMENT_ID
        JOIN PROD.DWH.D_CATEGORY dc
            ON dc.CATEGORY_ID = dt.CATEGORY_ID
        JOIN PROD.DWH.D_SPORT ds
            ON ds.SPORT_ID = dc.SPORT_ID
    WHERE de.EVENT_START_DT BETWEEN CURRENT_TIMESTAMP - INTERVAL '6 HOUR'
                                AND CURRENT_TIMESTAMP + INTERVAL '24 HOUR'
),
base AS (
    /* one row per (normalised) tournament × event × outcome × band */
    SELECT
        IFF(ter.TOURNAMENT_NAME ILIKE '%World Cup 2026%',
            'World Cup 2026', ter.TOURNAMENT_NAME)            AS TOURNAMENT_NAME,
        ter.EVENT_NAME,
        ter.OUTCOME,
        LEAST(FLOOR((ter.MINUTES_AFTER_START + 60) / 10) + 1, 16) AS BAND_ORDER,
        SUM(ter.GGR_EUR)   AS GGR_EUR,
        SUM(ter.N_PLAYERS) AS N_PLAYERS
    FROM PROD.SBX_BI.DD_HISTORY_Long_TERM ter
    JOIN tournament t ON t.TOURNAMENT_NAME = ter.TOURNAMENT_NAME
    WHERE ter.EVENT_START_DT >= DATEADD(DAY, -30, CURRENT_DATE())
      AND ter.MINUTES_AFTER_START BETWEEN -60 AND 99
    GROUP BY 1, 2, 3, 4
),
cum AS (
    /* running totals per event × outcome across bands (window functions) */
    SELECT
        b.*,
        SUM(b.GGR_EUR)   OVER (PARTITION BY b.TOURNAMENT_NAME, b.EVENT_NAME, b.OUTCOME
                               ORDER BY b.BAND_ORDER)          AS CUM_GGR_EUR,
        SUM(b.N_PLAYERS) OVER (PARTITION BY b.TOURNAMENT_NAME, b.EVENT_NAME, b.OUTCOME
                               ORDER BY b.BAND_ORDER)          AS CUM_N_PLAYERS
    FROM base b
),
per_outcome AS (
    SELECT
        TOURNAMENT_NAME,
        BAND_ORDER,
        OUTCOME,
        MIN(GGR_EUR)     AS MIN_GGR_EUR_STAKE,
        MAX(GGR_EUR)     AS MAX_GGR_EUR_STAKE,
        MAX(CUM_GGR_EUR) AS MAX_CUM_GGR_EUR_STAKE,
        MIN(CUM_GGR_EUR) AS MIN_CUM_GGR_EUR_STAKE
    FROM cum
    GROUP BY 1, 2, 3
),
players AS (
    /* max cumulative players per tournament × band, across events & outcomes
       (same shape as the old max_cum_players merge) */
    SELECT
        TOURNAMENT_NAME,
        BAND_ORDER,
        MAX(CUM_N_PLAYERS) AS MAX_CUM_N_PLAYERS
    FROM cum
    GROUP BY 1, 2
)
SELECT
    p.TOURNAMENT_NAME,
    p.BAND_ORDER,
    p.OUTCOME,
    p.MIN_GGR_EUR_STAKE,
    p.MAX_GGR_EUR_STAKE,
    p.MAX_CUM_GGR_EUR_STAKE,
    p.MIN_CUM_GGR_EUR_STAKE,
    pl.MAX_CUM_N_PLAYERS
FROM per_outcome p
JOIN players pl
  ON pl.TOURNAMENT_NAME = p.TOURNAMENT_NAME
 AND pl.BAND_ORDER      = p.BAND_ORDER
ORDER BY p.TOURNAMENT_NAME, p.BAND_ORDER, p.OUTCOME
"""

df_tournament_history = session.sql(QUERY).to_pandas()

# ── Light-touch pandas: labels, types, rounding (frame is tiny now) ──────────
BAND_LABELS = {
    1: '−60 to −51 min',  2: '−50 to −41 min',  3: '−40 to −31 min',
    4: '−30 to −21 min',  5: '−20 to −11 min',  6: '−10 to  −1 min',
    7: '0 to  +9 min',    8: '+10 to +19 min',  9: '+20 to +29 min',
    10: '+30 to +39 min', 11: '+40 to +49 min', 12: '+50 to +59 min',
    13: '+60 to +69 min', 14: '+70 to +79 min', 15: '+80 to +89 min',
    16: '+90 to +99 min',
}
df_tournament_history['BAND_ORDER'] = df_tournament_history['BAND_ORDER'].astype(int)
df_tournament_history['BAND_LABEL'] = df_tournament_history['BAND_ORDER'].map(BAND_LABELS)
df_tournament_history['OUTCOME'] = df_tournament_history['OUTCOME'].astype(str)

for _c in ['MIN_GGR_EUR_STAKE', 'MAX_GGR_EUR_STAKE',
           'MAX_CUM_GGR_EUR_STAKE', 'MIN_CUM_GGR_EUR_STAKE']:
    df_tournament_history[_c] = pd.to_numeric(df_tournament_history[_c], errors='coerce').round(2)
df_tournament_history['MAX_CUM_N_PLAYERS'] = (
    pd.to_numeric(df_tournament_history['MAX_CUM_N_PLAYERS'], errors='coerce')
      .round(0).astype('Int64')
)

# ── Final output: one unique row per TOURNAMENT / BAND / OUTCOME ─────────────
df_tournament_history = df_tournament_history[[
    'TOURNAMENT_NAME',
    'BAND_ORDER',
    'BAND_LABEL',
    'OUTCOME',
    'MIN_GGR_EUR_STAKE',
    'MAX_GGR_EUR_STAKE',
    'MAX_CUM_GGR_EUR_STAKE',
    'MIN_CUM_GGR_EUR_STAKE',
    'MAX_CUM_N_PLAYERS',
]].reset_index(drop=True)

print(f"Result shape : {df_tournament_history.shape[0]} rows × {df_tournament_history.shape[1]} columns")
print(f"Tournaments  : {df_tournament_history['TOURNAMENT_NAME'].nunique()}")
print(f"Bands        : {df_tournament_history['BAND_ORDER'].nunique()} (1–16)")

# Preview only — full frame lives in df_tournament_history for downstream use
display(df_tournament_history.head(100))

In [ ]:
import gc
import pandas as pd
import numpy as np
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# ── Memory-optimised pull ─────────────────────────────────────────────────────
# The previous version fetched RAW rows (every ticket-band-outcome row, 180
# days) and did banding + GGR_PLAYER + aggregation in pandas → kernel OOM.
# This version pushes all row-level work into SQL and returns ONE row per
# EVENT × OUTCOME × BAND (AMENDED: OUTCOME added so per-outcome odds can be
# derived), already carrying:
#   - band order (FLOOR-based, same 16-band scheme)
#   - summed GGR_EUR / N_PLAYERS / STAKE_EUR
# NOTE (AMENDED): the old row-level GGR_PLAYER column
#   (N_PLAYERS * ABS(GGR_EUR/STAKE_EUR + 1)) is no longer needed — the
#   CUM_N_PLAYERS metric is now built in pandas as
#   CUM_N_PLAYERS × (CUM_GGR_EUR / CUM_STAKE_EUR + 1) per event × band.
QUERY = """
with tournament as (
    SELECT DISTINCT dt.TOURNAMENT_NAME,
           ds.SPORT_NAME
    FROM PROD.DWH.D_EVENT de
        JOIN PROD.DWH.D_TOURNAMENT dt
            ON dt.TOURNAMENT_ID = de.TOURNAMENT_ID
        JOIN PROD.DWH.D_CATEGORY dc
            ON dc.CATEGORY_ID = dt.CATEGORY_ID
        JOIN PROD.DWH.D_SPORT ds
            ON ds.SPORT_ID = dc.SPORT_ID
    WHERE de.EVENT_START_DT BETWEEN CURRENT_TIMESTAMP - INTERVAL '4 HOUR'
                                AND CURRENT_TIMESTAMP + INTERVAL '12 HOUR'
)
SELECT
    TEAM1,
    TEAM2,
    EVENT_NAME,
    OUTCOME,
    /* 16-band scheme: band 1 = -60..-51, ..., band 16 = +90..+99 */
    LEAST(FLOOR((MINUTES_AFTER_START + 60) / 10) + 1, 16) AS BAND_ORDER,
    SUM(GGR_EUR)    AS GGR_EUR,
    SUM(N_PLAYERS)  AS N_PLAYERS,
    SUM(STAKE_EUR)  AS STAKE_EUR
FROM PROD.SBX_BI.DD_HISTORY_Long_TERM ter
JOIN tournament t ON t.TOURNAMENT_NAME = ter.TOURNAMENT_NAME
WHERE EVENT_START_DT >= DATEADD(DAY, -180, CURRENT_DATE())
  AND MINUTES_AFTER_START BETWEEN -60 AND 99
  AND TEAM1 IS NOT NULL
  AND TEAM2 IS NOT NULL
GROUP BY TEAM1, TEAM2, EVENT_NAME, OUTCOME,
         LEAST(FLOOR((MINUTES_AFTER_START + 60) / 10) + 1, 16)
"""

df_raw = session.sql(QUERY).to_pandas()

# Downcast numerics — halves memory for the value columns
for _c in ['GGR_EUR', 'N_PLAYERS', 'STAKE_EUR']:
    df_raw[_c] = pd.to_numeric(df_raw[_c], errors='coerce').astype('float32')
df_raw['BAND_ORDER'] = df_raw['BAND_ORDER'].astype('int8')
df_raw['OUTCOME'] = df_raw['OUTCOME'].astype(str)

BAND_LABELS = {
    1: '−60 to −51 min',  2: '−50 to −41 min',  3: '−40 to −31 min',
    4: '−30 to −21 min',  5: '−20 to −11 min',  6: '−10 to  −1 min',
    7: '0 to  +9 min',    8: '+10 to +19 min',  9: '+20 to +29 min',
    10: '+30 to +39 min', 11: '+40 to +49 min', 12: '+50 to +59 min',
    13: '+60 to +69 min', 14: '+70 to +79 min', 15: '+80 to +89 min',
    16: '+90 to +99 min',
}

# ─── AVG DELTA ODDS per team × band × outcome (NEW) ───────────────────────────
# Implied odds per aggregate:  ODDS = 1 − SUM(GGR_EUR) / SUM(STAKE_EUR)
# Per event × outcome (1/X/2):
#   ODDS_PRE  = implied odds over the whole pre-match window (bands 1–6,
#               GGR and STAKE summed across those bands) — the "Average
#               Pre-Game odds" of the event.
#   ODDS_BAND = implied odds within each single band — the "last odds"
#               observed when the game sits at that band.
#   DELTA_ODDS = ODDS_BAND / ODDS_PRE  (same ratio convention as the model
#               notebook's DELTA_ODDS fields: >1 drifted longer, <1 shortened).
# The deltas are then averaged across ALL the team's historical events →
# AVG_DELTA_ODDS_1 / AVG_DELTA_ODDS_X / AVG_DELTA_ODDS_2 per TEAM × BAND.
# Guards: bands/windows with zero stake yield NaN odds; events whose pre-game
# odds are missing or zero are excluded from the average.
_out = df_raw[df_raw['OUTCOME'].isin(['1', 'X', '2'])].copy()

_pre = (
    _out[_out['BAND_ORDER'] <= 6]
    .groupby(['EVENT_NAME', 'OUTCOME'], observed=True)
    .agg(_GGR=('GGR_EUR', 'sum'), _STK=('STAKE_EUR', 'sum'))
    .reset_index()
)
_pre['ODDS_PRE'] = np.where(_pre['_STK'] > 0, 1.0 - _pre['_GGR'] / _pre['_STK'], np.nan)

_out['ODDS_BAND'] = np.where(_out['STAKE_EUR'] > 0, 1.0 - _out['GGR_EUR'] / _out['STAKE_EUR'], np.nan)
_out = _out.merge(_pre[['EVENT_NAME', 'OUTCOME', 'ODDS_PRE']], on=['EVENT_NAME', 'OUTCOME'], how='left')
_out['DELTA_ODDS'] = np.where(
    pd.notna(_out['ODDS_BAND']) & pd.notna(_out['ODDS_PRE']) & (np.abs(_out['ODDS_PRE']) > 0),
    _out['ODDS_BAND'] / _out['ODDS_PRE'],
    np.nan
)

# Unpivot to team level — each event contributes to both teams' histories
_team_delta = pd.concat(
    [_out[['TEAM1', 'BAND_ORDER', 'OUTCOME', 'DELTA_ODDS']].rename(columns={'TEAM1': 'TEAM'}),
     _out[['TEAM2', 'BAND_ORDER', 'OUTCOME', 'DELTA_ODDS']].rename(columns={'TEAM2': 'TEAM'})],
    ignore_index=True
)
_team_delta = _team_delta[_team_delta['TEAM'].notna() & (_team_delta['TEAM'].str.strip() != '')]

avg_delta = (
    _team_delta.dropna(subset=['DELTA_ODDS'])
    .groupby(['TEAM', 'BAND_ORDER', 'OUTCOME'], observed=True)['DELTA_ODDS']
    .mean()
    .unstack('OUTCOME')
    .rename(columns={'1': 'AVG_DELTA_ODDS_1', 'X': 'AVG_DELTA_ODDS_X', '2': 'AVG_DELTA_ODDS_2'})
    .reset_index()
)
for _c in ['AVG_DELTA_ODDS_1', 'AVG_DELTA_ODDS_X', 'AVG_DELTA_ODDS_2']:
    if _c not in avg_delta.columns:
        avg_delta[_c] = np.nan
avg_delta['TEAM'] = avg_delta['TEAM'].astype(str)

del _out, _pre, _team_delta
gc.collect()

# ─── Collapse to EVENT × BAND for the existing pipeline (outcome summed out) ──
df_ev = (
    df_raw
    .groupby(['TEAM1', 'TEAM2', 'EVENT_NAME', 'BAND_ORDER'], observed=True)
    [['GGR_EUR', 'N_PLAYERS', 'STAKE_EUR']].sum()
    .reset_index()
)
del df_raw
gc.collect()

# ─── Cumulative curves per event (event-level, pre-unpivot) ───────────────────
df_ev = df_ev.sort_values(['EVENT_NAME', 'BAND_ORDER'])
df_ev['CUM_N_PLAYERS'] = df_ev.groupby('EVENT_NAME')['N_PLAYERS'].cumsum()
df_ev['CUM_GGR_EUR_EVT']   = df_ev.groupby('EVENT_NAME')['GGR_EUR'].cumsum()
df_ev['CUM_STAKE_EUR_EVT'] = df_ev.groupby('EVENT_NAME')['STAKE_EUR'].cumsum()

# ── CUM_N_PLAYERS_ADJ (AMENDED definition) ────────────────────────────────────
# Cum N Players = cumulative N_PLAYERS at each band × ABS(GGR/STAKE + 1),
# where GGR/STAKE is the event's cumulative margin up to that band
# (CUM_GGR_EUR / CUM_STAKE_EUR). ABS keeps the multiplier non-negative even
# when the cumulative margin is below −100%. Falls back to the raw cumulative
# player count when cumulative stake is 0 (multiplier → 1).
df_ev['CUM_N_PLAYERS_ADJ'] = np.where(
    df_ev['CUM_STAKE_EUR_EVT'].abs() > 0,
    df_ev['CUM_N_PLAYERS'] * np.abs(df_ev['CUM_GGR_EUR_EVT'] / df_ev['CUM_STAKE_EUR_EVT'] + 1),
    df_ev['CUM_N_PLAYERS']
).astype('float32')

# ─── Unpivot — one row per TEAM per event × band ─────────────────────────────
base_cols = ['BAND_ORDER', 'GGR_EUR', 'STAKE_EUR', 'N_PLAYERS',
             'CUM_N_PLAYERS', 'CUM_N_PLAYERS_ADJ', 'EVENT_NAME']

df_teams = pd.concat(
    [df_ev[base_cols + ['TEAM1']].rename(columns={'TEAM1': 'TEAM'}),
     df_ev[base_cols + ['TEAM2']].rename(columns={'TEAM2': 'TEAM'})],
    ignore_index=True
)
df_teams = df_teams[df_teams['TEAM'].notna() & (df_teams['TEAM'].str.strip() != '')]
df_teams['TEAM'] = df_teams['TEAM'].astype('category')   # big win: repeated strings → codes

del df_ev
gc.collect()

# ─── TEAM × EVENT × BAND level (already aggregated by SQL; just cumsums) ─────
ev_band = df_teams[['TEAM', 'EVENT_NAME', 'BAND_ORDER', 'GGR_EUR', 'STAKE_EUR']].copy()
ev_band = ev_band.sort_values(['TEAM', 'EVENT_NAME', 'BAND_ORDER'])
ev_band['CUM_GGR_EUR'] = ev_band.groupby(['TEAM', 'EVENT_NAME'], observed=True)['GGR_EUR'].cumsum()
ev_band['CUM_STAKE_EUR'] = ev_band.groupby(['TEAM', 'EVENT_NAME'], observed=True)['STAKE_EUR'].cumsum()

# ─── Reference events per team ────────────────────────────────────────────────
ev_final = (
    ev_band
    .groupby(['TEAM', 'EVENT_NAME'], observed=True)
    .agg(
        FINAL_GGR_EUR = ('GGR_EUR', 'sum'),
    )
    .reset_index()
)

worst_ggr_event = (
    ev_final.loc[ev_final.groupby('TEAM', observed=True)['FINAL_GGR_EUR'].idxmin(),
                 ['TEAM', 'EVENT_NAME']]
    .rename(columns={'EVENT_NAME': 'REF_EVENT'})
)
best_ggr_event = (
    ev_final.loc[ev_final.groupby('TEAM', observed=True)['FINAL_GGR_EUR'].idxmax(),
                 ['TEAM', 'EVENT_NAME']]
    .rename(columns={'EVENT_NAME': 'REF_EVENT'})
)

del ev_final
gc.collect()

# ─── Team × band skeleton and curve extraction ────────────────────────────────
skeleton = (
    df_teams[['TEAM', 'BAND_ORDER']]
    .drop_duplicates()
    .sort_values(['TEAM', 'BAND_ORDER'])
    .reset_index(drop=True)
)

def event_curve(ref_events, value_col, out_col, cumulative):
    """
    Project one reference event per team onto the team × band skeleton.
    - Per-band values (cumulative=False): NaN where the event had no rows in a band.
    - Cumulative values (cumulative=True): forward-filled across bands, leading
      gaps filled with 0 (nothing accumulated yet).
    """
    curve = (
        skeleton
        .merge(ref_events, on='TEAM', how='left')
        .merge(
            ev_band[['TEAM', 'EVENT_NAME', 'BAND_ORDER', value_col]],
            left_on=['TEAM', 'REF_EVENT', 'BAND_ORDER'],
            right_on=['TEAM', 'EVENT_NAME', 'BAND_ORDER'],
            how='left'
        )
        .sort_values(['TEAM', 'BAND_ORDER'])
    )
    if cumulative:
        curve[value_col] = curve.groupby('TEAM', observed=True)[value_col].ffill().fillna(0.0)
    return curve[value_col].rename(out_col).reset_index(drop=True)

df_team_history = skeleton.copy()
df_team_history['BAND_LABEL'] = df_team_history['BAND_ORDER'].map(BAND_LABELS)

# Worst-GGR event: per-band values + cumulative curve
df_team_history['MIN_GGR_EUR_STAKE']     = event_curve(worst_ggr_event, 'GGR_EUR',     'MIN_GGR_EUR_STAKE',     cumulative=False)
df_team_history['MIN_CUM_GGR_EUR_STAKE'] = event_curve(worst_ggr_event, 'CUM_GGR_EUR', 'MIN_CUM_GGR_EUR_STAKE', cumulative=True)

# Best-GGR event: mirror
df_team_history['MAX_GGR_EUR_STAKE']     = event_curve(best_ggr_event, 'GGR_EUR',     'MAX_GGR_EUR_STAKE',     cumulative=False)
df_team_history['MAX_CUM_GGR_EUR_STAKE'] = event_curve(best_ggr_event, 'CUM_GGR_EUR', 'MAX_CUM_GGR_EUR_STAKE', cumulative=True)

# ── MAX_CUM_GGR_N_PLAYERS (NEW) ───────────────────────────────────────────────
# MAX_CUM_N_PLAYERS × (MAX_CUM_GGR_EUR_STAKE / MAX_CUM_EUR_STAKE + 1) per band.
# MAX_CUM_EUR_STAKE = cumulative STAKE_EUR curve of the same best-GGR reference
# event (helper column, not kept in the final output). Falls back to
# MAX_CUM_N_PLAYERS when cumulative stake is 0 (multiplier → 1).
df_team_history['MAX_CUM_EUR_STAKE'] = event_curve(best_ggr_event, 'CUM_STAKE_EUR', 'MAX_CUM_EUR_STAKE', cumulative=True)

# ── MAX_CUM_N_PLAYERS / MIN_CUM_N_PLAYERS (AMENDED definition) ───────────────
# Cum N Players = cumulative N_PLAYERS at each band × ABS(GGR/STAKE + 1)
# (CUM_N_PLAYERS_ADJ, built pre-unpivot at event level).
#   MAX = upper envelope across ALL the team's events: per-band max of
#         CUM_N_PLAYERS_ADJ, then a running cummax so the curve stays
#         monotone non-decreasing across bands.
#   MIN = lower envelope across ALL the team's events: per-band min of
#         CUM_N_PLAYERS_ADJ, then a running cummin (floor curve).
# This replaces the old 3-stage GGR_PLAYER reference-event construction for
# MAX and the raw CUM_N_PLAYERS envelope for MIN.
max_cum_players = (
    df_teams
    .groupby(['TEAM', 'BAND_ORDER'], observed=True)['CUM_N_PLAYERS_ADJ'].max()
    .groupby('TEAM', observed=True).cummax()
    .rename('MAX_CUM_N_PLAYERS')
    .reset_index()
)
df_team_history = df_team_history.merge(max_cum_players, on=['TEAM', 'BAND_ORDER'], how='left')

min_cum_players = (
    df_teams
    .groupby(['TEAM', 'BAND_ORDER'], observed=True)['CUM_N_PLAYERS_ADJ'].min()
    .groupby('TEAM', observed=True).cummin()
    .rename('MIN_CUM_N_PLAYERS')
    .reset_index()
)
df_team_history = df_team_history.merge(min_cum_players, on=['TEAM', 'BAND_ORDER'], how='left')

df_team_history['MAX_CUM_GGR_N_PLAYERS'] = np.where(
    df_team_history['MAX_CUM_EUR_STAKE'].abs() > 0,
    df_team_history['MAX_CUM_N_PLAYERS']
        * (1 - df_team_history['MAX_CUM_GGR_EUR_STAKE'] / df_team_history['MAX_CUM_EUR_STAKE']),
    df_team_history['MAX_CUM_N_PLAYERS']
)

del df_teams, ev_band, max_cum_players, min_cum_players
gc.collect()

# ─── Merge AVG_DELTA_ODDS onto the team × band history (NEW) ─────────────────
df_team_history['TEAM'] = df_team_history['TEAM'].astype(str)   # back to plain strings for downstream lookups
df_team_history = df_team_history.merge(avg_delta, on=['TEAM', 'BAND_ORDER'], how='left')

del avg_delta
gc.collect()

# ─── Column order + rounding ──────────────────────────────────────────────────
df_team_history = df_team_history[[
    'TEAM', 'BAND_ORDER', 'BAND_LABEL',
    'MIN_GGR_EUR_STAKE', 'MAX_GGR_EUR_STAKE',
    'MAX_CUM_N_PLAYERS', 'MIN_CUM_N_PLAYERS',
    'MAX_CUM_GGR_EUR_STAKE', 'MIN_CUM_GGR_EUR_STAKE',
    'MAX_CUM_GGR_N_PLAYERS',
    'AVG_DELTA_ODDS_1', 'AVG_DELTA_ODDS_X', 'AVG_DELTA_ODDS_2',
]].sort_values(['TEAM', 'BAND_ORDER']).reset_index(drop=True)

for _c in ['MIN_GGR_EUR_STAKE', 'MAX_GGR_EUR_STAKE',
           'MAX_CUM_GGR_EUR_STAKE', 'MIN_CUM_GGR_EUR_STAKE',
           'MIN_CUM_N_PLAYERS', 'MAX_CUM_N_PLAYERS',
           'MAX_CUM_GGR_N_PLAYERS']:
    df_team_history[_c] = df_team_history[_c].astype('float64').round(2)

for _c in ['AVG_DELTA_ODDS_1', 'AVG_DELTA_ODDS_X', 'AVG_DELTA_ODDS_2']:
    df_team_history[_c] = df_team_history[_c].astype('float64').round(4)

# ─── Summary (capped output) ──────────────────────────────────────────────────
print(f"Result shape : {df_team_history.shape[0]} rows × {df_team_history.shape[1]} columns")
print(f"Teams        : {df_team_history['TEAM'].nunique()}")
print(f"Bands        : {df_team_history['BAND_ORDER'].nunique()} (1–16)")
print()

# Preview only — full frame lives in df_team_history for downstream use
display(df_team_history.head(100))

In [ ]:
%%sql -r dataframe_2
SELECT COMPETITION, CLUB, PLAYER_NAME, CURRENT_VALUE
FROM PROD.SBX_BI.DD_VALUE_TABLE

In [ ]:
%%sql -r dataframe_3
SELECT EVENT_DATE, EVENT_NAME, MAX(CAST(EVENT_START_DT as date)) as EVENT_START, TOURNAMENT_NAME, MARKET_NAME, SPORT_NAME, TEAM1, TEAM2, MIN(MINUTES_AFTER_START) as MINUTES_AFTER_START, ROUND(SUM(stake_eur),0) as stake_eur, ROUND(SUM(GGR_EUR),0) as GGR   , SUM(N_TICKETS) as N_TICKETS  FROM PROD.SBX_BI.DD_HISTORY_Long_TERM WHERE 
EVENT_DATE BETWEEN CURRENT_DATE -10 and CURRENT_DATE 
--AND ACCEPTED_DT BETWEEN EVENT_START_DT - interval '60 minute' 
--and EVENT_START_DT  + Interval '20 Minute'

GROUP BY ALL ;

In [ ]:
# ─── STEP 9f: Tennis fundamentals — model divergence signal ──────────────────
# Joins PROD.SBX_BI.DD_TENNIS_FUNDAMENT_MODEL to live tennis events in `result`
# (run AFTER the snapshot-model cell; follows the 9b/9c/9e join pattern:
#  normalise -> exact -> structured fallback -> fuzzy, with match-type/score
#  columns and a try/except so a missing source never kills the run).
#
# Signal: FUND_DIVERGENCE_1 = CUM_P_1(live, last snapshot) - FUND_P_MODEL_1
#   > 0  => live betting flow leans toward side 1 MORE than fundamentals imply
#   (2-way sport: FUND_DIVERGENCE_2 = -FUND_DIVERGENCE_1, X undefined)
#
# Fundament rows are PLAYED matches, so the join is two-tier and honest about it:
#   FIXTURE       same pair within +/- FIXTURE_WINDOW_DAYS of the event
#   LAST_MEETING  most recent prior meeting of the pair (flagged, with age)
#   NO_MATCH      pair never met in the table / players unresolved
import re
import unicodedata
import difflib
import pandas as pd
import numpy as np
from collections import defaultdict

FIXTURE_WINDOW_DAYS   = 3
LAST_MEETING_MAX_DAYS = 540        # ignore meetings older than ~18 months
PLAYER_FUZZY_CUTOFF   = 0.85

try:
    # ── load fundamentals (small table: pull needed columns only) ──────────
    _fund = session.sql("""
        SELECT MATCH_DATE, P1, P2, TOUR, SURFACE, P_MODEL, P_MARKET
        FROM PROD.SBX_BI.DD_TENNIS_FUNDAMENT_MODEL
    """).to_pandas()
    _fund.columns = [c.upper() for c in _fund.columns]
    _fund['MATCH_DATE'] = pd.to_datetime(_fund['MATCH_DATE'])
    for c in ('P_MODEL', 'P_MARKET'):
        _fund[c] = pd.to_numeric(_fund[c], errors='coerce')

    # ── player resolver: live name -> fundament canonical name ────────────
    _players = pd.unique(pd.concat([_fund['P1'], _fund['P2']]))
    _canon_set = set(_players)
    _by_surname = defaultdict(list)
    for _p in _players:                       # index 1/2/3-token surname prefixes
        _t = _p.split(' ')
        for _k in range(1, min(3, len(_t)) + 1):
            _by_surname[' '.join(_t[:_k])].append(_p)

    def _tennis_norm(s):
        if s is None or (isinstance(s, float) and pd.isna(s)):
            return None
        s = unicodedata.normalize('NFD', str(s))
        s = ''.join(ch for ch in s if unicodedata.category(ch) != 'Mn')
        s = s.replace('.', ' ').replace('-', ' ').replace(',', ' ')
        return re.sub(r'\s+', ' ', s).strip().upper()

    def _initial_hits(surname, firstname):
        cands = _by_surname.get(surname, [])
        st = len(surname.split(' '))
        hits = [c for c in cands
                if len(c.split(' ')) > st and c.split(' ')[st][0] == firstname[0]]
        if len(hits) > 1:                     # PLISKOVA KR vs KA, WANG XIN vs XIY
            finer = [c for c in hits if firstname.startswith(c.split(' ')[st])]
            if len(finer) == 1:
                return finer
        return hits

    def _resolve_player(raw):
        n = _tennis_norm(raw)
        if not n:
            return None, 'NONE', None
        if n in _canon_set:
            return n, 'EXACT', 1.0
        toks = n.split(' ')
        if len(toks) >= 2:
            keys = []
            for k in range(1, min(4, len(toks))):
                keys.append((' '.join(toks[-k:]), toks[0]))   # 'First ... Last(s)'
                keys.append((' '.join(toks[:k]),  toks[k]))   # 'Last(s) First/init'
            for sur, fn in keys:
                hits = _initial_hits(sur, fn)
                if len(hits) == 1:
                    return hits[0], 'SURNAME_INITIAL', 1.0
            for sur, fn in keys:
                if len(_initial_hits(sur, fn)) > 1:
                    return None, 'AMBIGUOUS', None
        close = difflib.get_close_matches(n, list(_canon_set), n=1,
                                          cutoff=PLAYER_FUZZY_CUTOFF)
        if close:
            score = difflib.SequenceMatcher(None, n, close[0]).ratio()
            return close[0], 'FUZZY', round(score, 3)
        return None, 'NO_MATCH', None

    # ── pair index: frozenset(P1,P2) -> fundament rows ─────────────────────
    _pair_rows = defaultdict(list)
    for _i, _r in _fund.iterrows():
        _pair_rows[frozenset((_r['P1'], _r['P2']))].append(_i)

    def _lookup_pair(c1, c2, event_date):
        idxs = _pair_rows.get(frozenset((c1, c2)))
        if not idxs:
            return None, 'NO_MATCH', None
        sub = _fund.loc[idxs]
        off = (sub['MATCH_DATE'] - event_date).dt.days
        near = off.abs().idxmin()
        if abs(off.loc[near]) <= FIXTURE_WINDOW_DAYS:
            return near, 'FIXTURE', int(off.loc[near])
        past = sub[sub['MATCH_DATE'] < event_date]
        if past.empty:
            return None, 'NO_MATCH', None
        last = past['MATCH_DATE'].idxmax()
        age = int((event_date - past.loc[last, 'MATCH_DATE']).days)
        if age > LAST_MEETING_MAX_DAYS:
            return None, 'NO_MATCH', None
        return last, 'LAST_MEETING', -age

    # ── live side-1 cumulative probability (2-way: from PRED + PRED_PROB) ──
    def _live_cum_p1(r):
        pred, prob = r.get('M_LAST_SNAPSHOT_PRED'), r.get('M_LAST_SNAPSHOT_PRED_PROB')
        if pred is None or prob is None or pd.isna(prob):
            return np.nan
        pred = str(pred)
        if pred == '1':
            return float(prob)
        if pred == '2':
            return 1.0 - float(prob)
        return np.nan                          # 'X' should not occur for tennis

    # ── walk tennis rows of `result` ───────────────────────────────────────
    _is_tennis = result['SPORT_NAME'].eq('Tennis')
    _ev_dt = (pd.to_datetime(result['REAL_START_TIME_PLUS_1H'], errors='coerce')
              - pd.Timedelta(hours=1))

    _cols = ['FUND_P1_NAME', 'FUND_P1_MATCH_TYPE', 'FUND_P1_MATCH_SCORE',
             'FUND_P2_NAME', 'FUND_P2_MATCH_TYPE', 'FUND_P2_MATCH_SCORE',
             'FUND_MATCH_TYPE', 'FUND_MATCH_DATE', 'FUND_DAYS_OFFSET',
             'FUND_TOUR', 'FUND_SURFACE',
             'FUND_P_MODEL_1', 'FUND_P_MARKET_1',
             'CUM_P_1_LIVE', 'FUND_DIVERGENCE_1', 'FUND_DIVERGENCE_2',
             'FUND_ABS_DIVERGENCE']
    _acc = {c: [] for c in _cols}

    for _idx, _r in result.iterrows():
        if not _is_tennis.loc[_idx]:
            for c in _cols:
                _acc[c].append(None)
            continue
        s1, s2 = _split_teams(_r.get('EVENT_NAME'))          # helper from 9e
        c1, m1, sc1 = _resolve_player(s1)
        c2, m2, sc2 = _resolve_player(s2)
        _acc['FUND_P1_NAME'].append(c1); _acc['FUND_P1_MATCH_TYPE'].append(m1)
        _acc['FUND_P1_MATCH_SCORE'].append(sc1)
        _acc['FUND_P2_NAME'].append(c2); _acc['FUND_P2_MATCH_TYPE'].append(m2)
        _acc['FUND_P2_MATCH_SCORE'].append(sc2)

        row_i, mtype, offset = (None, 'NO_MATCH', None)
        if c1 and c2 and pd.notna(_ev_dt.loc[_idx]):
            row_i, mtype, offset = _lookup_pair(c1, c2, _ev_dt.loc[_idx].normalize())
        _acc['FUND_MATCH_TYPE'].append(mtype)
        _acc['FUND_DAYS_OFFSET'].append(offset)

        if row_i is None:
            for c in ('FUND_MATCH_DATE', 'FUND_TOUR', 'FUND_SURFACE',
                      'FUND_P_MODEL_1', 'FUND_P_MARKET_1', 'CUM_P_1_LIVE',
                      'FUND_DIVERGENCE_1', 'FUND_DIVERGENCE_2',
                      'FUND_ABS_DIVERGENCE'):
                _acc[c].append(None)
            continue

        _f = _fund.loc[row_i]
        aligned = (_f['P1'] == c1)                     # orient to live side 1
        p_model_1 = float(_f['P_MODEL']) if aligned else 1.0 - float(_f['P_MODEL'])
        p_mkt = _f['P_MARKET']
        p_mkt_1 = (None if pd.isna(p_mkt)
                   else (float(p_mkt) if aligned else 1.0 - float(p_mkt)))
        cum_p1 = _live_cum_p1(_r)
        div1 = (round(cum_p1 - p_model_1, 4) if pd.notna(cum_p1) else None)

        _acc['FUND_MATCH_DATE'].append(_f['MATCH_DATE'].date())
        _acc['FUND_TOUR'].append(_f['TOUR'])
        _acc['FUND_SURFACE'].append(_f['SURFACE'])
        _acc['FUND_P_MODEL_1'].append(round(p_model_1, 4))
        _acc['FUND_P_MARKET_1'].append(None if p_mkt_1 is None else round(p_mkt_1, 4))
        _acc['CUM_P_1_LIVE'].append(None if pd.isna(cum_p1) else round(cum_p1, 4))
        _acc['FUND_DIVERGENCE_1'].append(div1)
        _acc['FUND_DIVERGENCE_2'].append(None if div1 is None else -div1)
        _acc['FUND_ABS_DIVERGENCE'].append(None if div1 is None else abs(div1))

    for c in _cols:
        result[c] = _acc[c]

    # ── report ─────────────────────────────────────────────────────────────
    _tn = result.loc[_is_tennis]
    print(f'STEP 9f: tennis fundamentals join — {len(_tn)} tennis events, '
          f'{len(_fund)} fundament rows, {len(_canon_set)} players')
    if len(_tn):
        print('  join type:', _tn['FUND_MATCH_TYPE'].value_counts(dropna=False).to_dict())
        print('  player resolution:',
              pd.concat([_tn['FUND_P1_MATCH_TYPE'], _tn['FUND_P2_MATCH_TYPE']])
                .value_counts(dropna=False).to_dict())
        _top = (_tn.dropna(subset=['FUND_ABS_DIVERGENCE'])
                   .nlargest(10, 'FUND_ABS_DIVERGENCE'))
        if len(_top):
            print('  top divergences (live cum P1 vs fundament P1):')
            for _, r in _top.iterrows():
                print(f"    {str(r['EVENT_NAME'])[:44]:44s} "
                      f"live {r['CUM_P_1_LIVE']:.3f} vs fund {r['FUND_P_MODEL_1']:.3f} "
                      f"-> div {r['FUND_DIVERGENCE_1']:+.3f} ({r['FUND_MATCH_TYPE']})")
except Exception as _e:
    print(f'STEP 9f skipped (tennis fundamentals unavailable): {_e}')
    for c in ['FUND_MATCH_TYPE', 'FUND_P_MODEL_1', 'FUND_DIVERGENCE_1',
              'FUND_DIVERGENCE_2', 'FUND_ABS_DIVERGENCE']:
        if c not in result.columns:
            result[c] = None

result[[c for c in ['REAL_START_TIME_PLUS_1H', 'EVENT_NAME', 'SPORT_NAME',
                    'FUND_MATCH_TYPE', 'FUND_P_MODEL_1', 'FUND_P_MARKET_1',
                    'CUM_P_1_LIVE', 'FUND_DIVERGENCE_1', 'FUND_ABS_DIVERGENCE']
        if c in result.columns]].loc[result['SPORT_NAME'].eq('Tennis')].head(20)

In [ ]:
# ─── STEP 9g: TENNIS RISK OUTCOME — trading data x fundamentals ──────────────
# Run AFTER STEP 9f. Builds `tennis_risk_signal`: one row per live tennis event
# combining the risk model's TRADING-DATA side (GGR flow by stake/player track,
# volumes, odds, live cumulative probability) with the FUNDAMENTALS side
# (DD_TENNIS_FUNDAMENT_MODEL probabilities joined in 9f), plus the divergence
# and a tiered ALERT flag. This is the integrated outcome of the two systems.
#
# Alert tiers (starting thresholds — calibrate after the first weeks of data):
#   FIXTURE rows       : |divergence| > 0.15
#   LAST_MEETING <=180d: |divergence| > 0.25
#   older / NO_MATCH   : never alerted on divergence (no reliable fundament)
import pandas as pd
import numpy as np

ALERT_THR_FIXTURE      = 0.15
ALERT_THR_LAST_MEETING = 0.25
LAST_MEETING_ALERT_MAX_AGE_DAYS = 180
WRITE_TO_TABLE = False                      # True -> publish to Snowflake below
SIGNAL_TABLE   = 'DD_TENNIS_RISK_SIGNAL'    # lands in PROD.SBX_BI

_tn = result.loc[result['SPORT_NAME'].eq('Tennis')].copy()

# ── trading-data side: flow metrics carried from the snapshot model ──────────
_TRADING_COLS = [
    'REAL_START_TIME_PLUS_1H', 'TOURNAMENT_NAME', 'EVENT_NAME',
    'N_PLAYERS_KO_WINDOW_10MIN', 'LAST_SNAPSHOT',
    'M_LAST_SNAPSHOT_PRED', 'M_LAST_SNAPSHOT_PRED_PROB',
    'SIGNAL_CONSISTENCY_PCT', 'PRED_STABILITY', 'SIGNAL_TREND',
    'M_LAST_SNAPSHOT_CUM_GGR_STAKE_1', 'M_LAST_SNAPSHOT_CUM_GGR_STAKE_2',
    'M_LAST_SNAPSHOT_CUM_GGR_PLAYER_1', 'M_LAST_SNAPSHOT_CUM_GGR_PLAYER_2',
    'KO_CUM_GGR_STAKE_1', 'KO_CUM_GGR_STAKE_2',
    'KO_CUM_GGR_PLAYER_1', 'KO_CUM_GGR_PLAYER_2',
    'AVG_ODDS_PRE_1', 'AVG_ODDS_PRE_2', 'LAST_ODDS_1', 'LAST_ODDS_2',
    'RATIO_GGR_PLAYER_TO_TOURNAMENT_MAX', 'RATIO_GGR_STAKE_TO_TOURNAMENT_MAX',
]
# ── fundamentals side: columns appended by STEP 9f ───────────────────────────
_FUND_COLS = [
    'FUND_P1_NAME', 'FUND_P2_NAME', 'FUND_MATCH_TYPE', 'FUND_MATCH_DATE',
    'FUND_DAYS_OFFSET', 'FUND_TOUR', 'FUND_SURFACE',
    'FUND_P_MODEL_1', 'FUND_P_MARKET_1',
    'CUM_P_1_LIVE', 'FUND_DIVERGENCE_1', 'FUND_DIVERGENCE_2',
    'FUND_ABS_DIVERGENCE',
]
keep = [c for c in _TRADING_COLS + _FUND_COLS if c in _tn.columns]
tennis_risk_signal = _tn[keep].copy()

# post-KO net stake-track flow imbalance toward side 1 (trading-data pressure)
for a, b, name in [('M_LAST_SNAPSHOT_CUM_GGR_STAKE_1', 'M_LAST_SNAPSHOT_CUM_GGR_STAKE_2',
                    'FLOW_IMBALANCE_STAKE'),
                   ('M_LAST_SNAPSHOT_CUM_GGR_PLAYER_1', 'M_LAST_SNAPSHOT_CUM_GGR_PLAYER_2',
                    'FLOW_IMBALANCE_PLAYER')]:
    if a in tennis_risk_signal.columns and b in tennis_risk_signal.columns:
        g1 = pd.to_numeric(tennis_risk_signal[a], errors='coerce')
        g2 = pd.to_numeric(tennis_risk_signal[b], errors='coerce')
        tot = g1.abs() + g2.abs()
        tennis_risk_signal[name] = ((g1 - g2) / tot.replace(0, np.nan)).round(4)

# implied market prob for side 1 from last live odds (trading-data market view)
if {'LAST_ODDS_1', 'LAST_ODDS_2'}.issubset(tennis_risk_signal.columns):
    o1 = pd.to_numeric(tennis_risk_signal['LAST_ODDS_1'], errors='coerce')
    o2 = pd.to_numeric(tennis_risk_signal['LAST_ODDS_2'], errors='coerce')
    inv1, inv2 = 1.0 / o1, 1.0 / o2
    tennis_risk_signal['LIVE_ODDS_P_1'] = (inv1 / (inv1 + inv2)).round(4)
    if 'FUND_P_MODEL_1' in tennis_risk_signal.columns:
        tennis_risk_signal['ODDS_VS_FUND_DIVERGENCE_1'] = (
            tennis_risk_signal['LIVE_ODDS_P_1']
            - pd.to_numeric(tennis_risk_signal['FUND_P_MODEL_1'], errors='coerce')
        ).round(4)

# ── tiered alert flag ────────────────────────────────────────────────────────
def _alert(r):
    div = r.get('FUND_ABS_DIVERGENCE')
    mt  = r.get('FUND_MATCH_TYPE')
    if div is None or pd.isna(div) or mt is None:
        return 'NO_FUNDAMENT'
    if mt == 'FIXTURE':
        return 'ALERT' if div > ALERT_THR_FIXTURE else 'OK'
    if mt == 'LAST_MEETING':
        off = r.get('FUND_DAYS_OFFSET')
        age = abs(off) if off is not None and pd.notna(off) else 10**6
        if age <= LAST_MEETING_ALERT_MAX_AGE_DAYS:
            return 'ALERT' if div > ALERT_THR_LAST_MEETING else 'OK'
        return 'STALE_FUNDAMENT'
    return 'NO_FUNDAMENT'

tennis_risk_signal['FUND_ALERT'] = tennis_risk_signal.apply(_alert, axis=1)
tennis_risk_signal = tennis_risk_signal.sort_values(
    'FUND_ABS_DIVERGENCE', ascending=False, na_position='last'
).reset_index(drop=True)

# ── report ───────────────────────────────────────────────────────────────────
print(f'STEP 9g: tennis risk outcome — {len(tennis_risk_signal)} events')
if len(tennis_risk_signal):
    print('  alert status:', tennis_risk_signal['FUND_ALERT'].value_counts().to_dict())
    _al = tennis_risk_signal[tennis_risk_signal['FUND_ALERT'] == 'ALERT']
    for _, r in _al.head(10).iterrows():
        print(f"  ALERT {str(r['EVENT_NAME'])[:42]:42s} "
              f"live {r['CUM_P_1_LIVE']} vs fund {r['FUND_P_MODEL_1']} "
              f"(div {r['FUND_DIVERGENCE_1']:+.3f}, {r['FUND_MATCH_TYPE']}, "
              f"stake-flow imb {r.get('FLOW_IMBALANCE_STAKE')})")

# ── optional publish of the integrated outcome ───────────────────────────────
if WRITE_TO_TABLE and len(tennis_risk_signal):
    from datetime import datetime, timezone
    _out = tennis_risk_signal.copy()
    _out['RUN_TS'] = datetime.now(timezone.utc)
    session.write_pandas(_out, table_name=SIGNAL_TABLE,
                         database='PROD', schema='SBX_BI',
                         auto_create_table=True, overwrite=True,
                         quote_identifiers=False, use_logical_type=True)
    print(f'written: PROD.SBX_BI.{SIGNAL_TABLE} ({len(_out)} rows)')

tennis_risk_signal.head(20)

## Performance Comparison: Raw.sql (optimised) vs Raw2.sql (original)

In [ ]:
%%sql -r tag1
-- Run the OPTIMISED version (Raw.sql)
ALTER SESSION SET QUERY_TAG = 'perf_test_raw_optimised';

In [ ]:
%%sql -r optimised_result
WITH max_accepted AS (
    SELECT MAX(fts.ACCEPTED_DT) AS mx
    FROM DWH.F_TICKET_SELECTION fts
    WHERE fts.IS_COUNT_RELEVANT = 1
      AND fts.ORIGINAL_RESOLVED_DT IS NULL
      AND fts.ACCEPTED_DT BETWEEN DATEADD(HOUR, -1, CURRENT_TIMESTAMP()) AND CURRENT_TIMESTAMP()
)
, tickets_filtered AS (
    SELECT A.RECORD_CONTENT, A.RECORD_METADATA, S.value AS sel_value
    FROM PROD.RAW_SPORTSBOOK.TICKETS_V2 A
       , max_accepted ma
       , LATERAL FLATTEN(input => A.RECORD_CONTENT:selections, OUTER => FALSE) AS S
    WHERE TO_TIMESTAMP_LTZ((A.RECORD_METADATA:CreateTime::INT) / 1000) >= ma.mx
      AND A.RECORD_CONTENT:selection_count_type::STRING <> 'MULTIPLE'
      AND TO_TIMESTAMP((A.RECORD_CONTENT:events::OBJECT[(S.value:event_id)]
              :real_start_time::INT) / 1000)
          BETWEEN CURRENT_TIMESTAMP - INTERVAL '3 HOURS'
              AND CURRENT_TIMESTAMP + INTERVAL '4 HOURS'
      AND RECORD_METADATA:headers:"betting.message_type":value = 'TICKET_APPROVED'
      AND S.value:odd."status"::VARCHAR = 'ACTIVE'
)
, flattened AS (
    SELECT
        LOWER(A.RECORD_CONTENT:id::VARCHAR) AS TICKET_ID,
        TO_TIMESTAMP_LTZ((A.RECORD_METADATA:CreateTime::INT) / 1000) AS CREATE_TIME,
        A.RECORD_CONTENT:target::VARCHAR AS BUSINESS_DOMAIN,
        A.RECORD_CONTENT:config.currency::VARCHAR AS CONFIG_CURRENCY,
        LOWER(A.RECORD_CONTENT:player_id::VARCHAR) AS PLAYER_ID,
        LOWER(A.sel_value:event_id::VARCHAR) AS EVENT_ID,
        TO_TIMESTAMP_LTZ(A.RECORD_CONTENT:events::OBJECT[(A.sel_value:event_id)]:real_start_time::BIGINT / 1000) AS REAL_START_TIME,
        f.value:key::VARCHAR AS SPECIFIER_KEY,
        f.value:value::VARCHAR AS SPECIFIER_VALUE,
        A.RECORD_CONTENT:payment.stake.total::FLOAT AS PAYMENT_STAKE_TOTAL,
        A.RECORD_CONTENT:initial_potential_payout.coefficient::FLOAT AS INITIALPOTENTIALPAYOUT_COEFFICIENT,
        LOWER(A.sel_value:id::VARCHAR) AS SELECTIONS_ID,
        A.sel_value:odd.coefficient::FLOAT AS SELECTIONS_ODD_COEFFICIENT,
        SPLIT_PART(A.sel_value:market_line.legacy_name, '|', 1)::VARCHAR AS SELECTIONS_MARKETLINE_LEGACY_NAME,
        SPLIT_PART(A.sel_value:odd.legacy_name, '|', 1)::VARCHAR AS SELECTIONS_ODD_LEGACY_NAME,
        A.sel_value:market_line."type"::VARCHAR AS SELECTIONS_MARKETLINE_TYPE
    FROM tickets_filtered A
       , LATERAL FLATTEN(input => A.sel_value:market_line.specifiers, OUTER => TRUE) AS f
)
, base AS (
    SELECT
        aa.TICKET_ID, aa.CREATE_TIME, aa.BUSINESS_DOMAIN, aa.CONFIG_CURRENCY,
        aa.PLAYER_ID, aa.EVENT_ID, aa.REAL_START_TIME, aa.PAYMENT_STAKE_TOTAL,
        aa.INITIALPOTENTIALPAYOUT_COEFFICIENT, aa.SELECTIONS_ID,
        aa.SELECTIONS_ODD_COEFFICIENT, aa.SELECTIONS_MARKETLINE_LEGACY_NAME,
        aa.SELECTIONS_ODD_LEGACY_NAME, aa.SELECTIONS_MARKETLINE_TYPE,
        de.EVENT_NAME, ds.SPORT_NAME, dt.TOURNAMENT_NAME,
        (aa.PAYMENT_STAKE_TOTAL * fx.MIDDLE_RATE)::FLOAT AS STAKE_EUR
    FROM flattened aa
    LEFT JOIN DWH.F_EXCHANGE_RATE fx
        ON fx.FROM_CURRENCY_CODE = aa.CONFIG_CURRENCY AND fx.TO_CURRENCY_CODE = 'EUR' AND fx.RATE_DATE = TO_DATE(aa.REAL_START_TIME)
    JOIN PROD.DWH.D_EVENT de ON de.EVENT_ID = aa.EVENT_ID
    JOIN PROD.DWH.D_TOURNAMENT dt ON dt.TOURNAMENT_ID = de.TOURNAMENT_ID
    JOIN PROD.DWH.D_CATEGORY dc ON dc.CATEGORY_ID = dt.CATEGORY_ID
    JOIN PROD.DWH.D_SPORT ds ON ds.SPORT_ID = dc.SPORT_ID
    WHERE de.EVENT_START_DT > CURRENT_TIMESTAMP - INTERVAL '5 HOUR'
      AND aa.SELECTIONS_MARKETLINE_LEGACY_NAME IN ('Money line (incl. OT)', 'Match Betting', 'Match Winner')
      AND ds.SPORT_NAME IN ('Soccer', 'Basketball', 'Tennis')
)
, base_windowed AS (
    SELECT b.*, DATEDIFF('second', b.REAL_START_TIME, b.CREATE_TIME) / 60.0 AS MINS_FROM_START
    FROM base b
    WHERE b.INITIALPOTENTIALPAYOUT_COEFFICIENT > 1.0
      AND b.STAKE_EUR > 0
      AND DATEDIFF('second', b.REAL_START_TIME, b.CREATE_TIME) / 60.0 BETWEEN -60 AND 240
)
, band_base AS (
    SELECT bw.EVENT_ID, bw.EVENT_NAME, bw.REAL_START_TIME, bw.TOURNAMENT_NAME,
        bw.BUSINESS_DOMAIN, bw.SELECTIONS_MARKETLINE_LEGACY_NAME AS MARKET_NAME, bw.SPORT_NAME,
        TRIM(SPLIT_PART(bw.EVENT_NAME, '·', 1)) AS team1,
        TRIM(SPLIT_PART(bw.EVENT_NAME, '·', 2)) AS team2,
        bw.SELECTIONS_ODD_LEGACY_NAME AS outcome,
        bw.SELECTIONS_ODD_COEFFICIENT AS odds, bw.STAKE_EUR AS stake_eur,
        bw.STAKE_EUR * bw.SELECTIONS_ODD_COEFFICIENT AS ggr_stake_row,
        bw.TICKET_ID AS PLAYER_ID,
        LEAST(FLOOR(bw.MINS_FROM_START / 10.0) * 10, 80) AS band_start_min,
        CONCAT(CAST(LEAST(FLOOR(bw.MINS_FROM_START / 10.0) * 10, 80) AS VARCHAR), ' to ',
               CAST(LEAST(FLOOR(bw.MINS_FROM_START / 10.0) * 10, 80) + 10 AS VARCHAR), ' mins') AS acceptance_time_band
    FROM base_windowed bw
)
, agg AS (
    SELECT EVENT_ID, EVENT_NAME, REAL_START_TIME, TOURNAMENT_NAME, BUSINESS_DOMAIN, MARKET_NAME, SPORT_NAME,
        team1, team2, outcome, acceptance_time_band, band_start_min,
        AVG(odds) AS odds, SUM(stake_eur) AS stake_eur, SUM(ggr_stake_row) AS ggr_stake_row,
        COUNT(DISTINCT PLAYER_ID) AS n_players, COUNT(DISTINCT PLAYER_ID) * AVG(odds) AS ggr_player_row
    FROM band_base
    GROUP BY EVENT_ID, EVENT_NAME, REAL_START_TIME, TOURNAMENT_NAME, BUSINESS_DOMAIN, MARKET_NAME, SPORT_NAME,
             team1, team2, outcome, acceptance_time_band, band_start_min
)
SELECT
    TO_CHAR(a.REAL_START_TIME + INTERVAL '1 HOUR', 'YYYY-MM-DD HH24:MI') AS REAL_START_TIME,
    a.EVENT_NAME, a.TOURNAMENT_NAME, a.BUSINESS_DOMAIN, a.MARKET_NAME, a.SPORT_NAME,
    a.team1 AS TEAM1, a.team2 AS TEAM2, a.outcome AS OUTCOME,
    a.acceptance_time_band AS ACCEPTANCE_TIME_BAND, a.odds,
    a.stake_eur AS STAKE_EUR, a.ggr_stake_row AS GGR_STAKE,
    a.n_players AS N_PLAYERS, a.ggr_player_row AS GGR_PLAYER,
    SUM(a.stake_eur) OVER (PARTITION BY a.EVENT_ID) AS STAKE_EUR_MATCH_TOTAL,
    SUM(a.n_players) OVER (PARTITION BY a.EVENT_ID) AS N_PLAYERS_MATCH_TOTAL
FROM agg a
ORDER BY REAL_START_TIME, a.band_start_min, STAKE_EUR DESC;

In [ ]:
%%sql -r tag2
-- Run the ORIGINAL version (Raw2.sql)
ALTER SESSION SET QUERY_TAG = 'perf_test_raw2_original';

In [ ]:
%%sql -r original_result
SELECT * FROM (
WITH tickets_filtered AS (
    SELECT *
    FROM PROD.RAW_SPORTSBOOK.TICKETS_V2 A
, LATERAL FLATTEN(input => A.RECORD_CONTENT:selections, OUTER => false) as S
    WHERE TO_TIMESTAMP_LTZ((RECORD_METADATA:CreateTime::INT) / 1000)
              >= (SELECT MAX(fts.ACCEPTED_DT) as mx
     FROM DWH.F_TICKET_SELECTION fts
      WHERE fts.IS_COUNT_RELEVANT    = 1
       AND fts.ORIGINAL_RESOLVED_DT IS NULL
      AND fts.ACCEPTED_DT          BETWEEN DATEADD(HOUR, -1, CURRENT_TIMESTAMP()) AND  CURRENT_TIMESTAMP()
      )
      AND A.RECORD_CONTENT:selection_count_type::STRING <> 'MULTIPLE'
      and to_timestamp((A.RECORD_CONTENT:events::OBJECT[(S.value:event_id )]:real_start_time::INT)/1000) between current_timestamp - interval '3 HOURS' and current_timestamp + INTERVAL '4 HOURS'
      AND RECORD_METADATA:headers:"betting.message_type":value = 'TICKET_APPROVED'
)
, flattened AS (
    SELECT
        LOWER(A.RECORD_CONTENT:id::VARCHAR) AS TICKET_ID,
        A.RECORD_CONTENT:"status"::VARCHAR AS TICKET_STATUS,
        TO_TIMESTAMP_LTZ((A.RECORD_METADATA:CreateTime::INT) / 1000) AS CREATE_TIME,
        A.RECORD_CONTENT:target::VARCHAR AS BUSINESS_DOMAIN,
        A.RECORD_CONTENT:config.currency::VARCHAR AS CONFIG_CURRENCY,
        LOWER(A.RECORD_CONTENT:player_id::VARCHAR) AS PLAYER_ID,
        A.RECORD_CONTENT:selection_count_type::VARCHAR AS SELECTION_COUNT_TYPE,
        LOWER(event.value:id::VARCHAR) AS EVENT_ID,
        event.value:name::VARCHAR AS EVENT_NAME,
        event.value:sport.name::VARCHAR AS SPORT_NAME,
        event.value:category.name::VARCHAR AS CATEGORY_NAME,
        event.value:tournament.name::VARCHAR AS TOURNAMENT_NAME,
        TO_TIMESTAMP_LTZ(event.value:real_start_time::BIGINT / 1000) AS REAL_START_TIME,
        f.value:key::VARCHAR AS SPECIFIER_KEY,
        f.value:value::VARCHAR AS SPECIFIER_VALUE,
        A.RECORD_CONTENT:payment.stake.total::FLOAT AS PAYMENT_STAKE_TOTAL,
        A.RECORD_CONTENT:initial_potential_payout.coefficient::FLOAT AS INITIALPOTENTIALPAYOUT_COEFFICIENT,
        A.RECORD_CONTENT:initial_potential_payout.potential_winning.amount::FLOAT AS INITIALPOTENTIALPAYOUT_POTENTIALWINNING_AMOUNT,
        A.RECORD_CONTENT:"system"."type"::VARCHAR AS SYSTEM_TYPE,
        A.RECORD_CONTENT:"system".combination_count::INT AS SYSTEM_COMBINATIONCOUNT,
        A.RECORD_CONTENT:event_type::VARCHAR AS EVENTS_TYPE,
        LOWER(S.value:id::VARCHAR) AS SELECTIONS_ID,
        S.value:"status"::VARCHAR AS SELECTIONS_STATUS,
        S.value:odd.coefficient::FLOAT AS SELECTIONS_ODD_COEFFICIENT,
        S.value:odd."status"::VARCHAR AS SELECTIONS_ODD_STATUS,
        SPLIT_PART(S.value:market_line.legacy_name, '|', 1)::VARCHAR AS SELECTIONS_MARKETLINE_LEGACY_NAME,
        SPLIT_PART(S.value:odd.legacy_name, '|', 1)::VARCHAR AS SELECTIONS_ODD_LEGACY_NAME,
        S.value:market_line."type"::VARCHAR AS SELECTIONS_MARKETLINE_TYPE
    FROM
        tickets_filtered A,
        LATERAL FLATTEN(input => A.RECORD_CONTENT:selections, OUTER => TRUE) AS S,
        LATERAL FLATTEN(input => S.value:market_line.specifiers, OUTER => TRUE) AS f,
        LATERAL FLATTEN(input => A.RECORD_CONTENT:events, OUTER => TRUE) AS event
    WHERE S.value:odd."status"::VARCHAR = 'ACTIVE'
)
, base AS (
    SELECT
        aa.* EXCLUDE (EVENT_NAME, SPORT_NAME, TOURNAMENT_NAME),
        de.EVENT_NAME, ds.SPORT_NAME, dt.TOURNAMENT_NAME,
        (aa.PAYMENT_STAKE_TOTAL * fx.MIDDLE_RATE)::FLOAT AS STAKE_EUR
    FROM flattened aa
    LEFT JOIN DWH.F_EXCHANGE_RATE fx
        ON fx.FROM_CURRENCY_CODE = aa.CONFIG_CURRENCY AND fx.TO_CURRENCY_CODE = 'EUR' AND fx.RATE_DATE = TO_DATE(aa.REAL_START_TIME)
    JOIN PROD.DWH.D_EVENT de ON de.EVENT_ID = aa.EVENT_ID
    JOIN PROD.DWH.D_TOURNAMENT dt ON dt.TOURNAMENT_ID = de.TOURNAMENT_ID
    JOIN PROD.DWH.D_CATEGORY dc ON dc.CATEGORY_ID = dt.CATEGORY_ID
    JOIN PROD.DWH.D_SPORT ds ON ds.SPORT_ID = dc.SPORT_ID
    WHERE de.EVENT_START_DT > CURRENT_TIMESTAMP - INTERVAL '5 HOUR'
        AND aa.SELECTIONS_MARKETLINE_LEGACY_NAME IN ('Money line (incl. OT)', 'Match Betting', 'Match Winner')
        AND ds.SPORT_NAME IN ('Soccer','Basketball', 'Tennis', 'Volleyball')
    GROUP BY ALL
)
, base_windowed AS (
    SELECT base.*,
        LAST_VALUE(base.INITIALPOTENTIALPAYOUT_COEFFICIENT IGNORE NULLS)
            OVER (PARTITION BY base.EVENT_ID, base.SELECTIONS_ODD_LEGACY_NAME,
                  base.SELECTIONS_MARKETLINE_LEGACY_NAME, base.SELECTIONS_MARKETLINE_TYPE
                  ORDER BY base.REAL_START_TIME) AS LAST_INITIALPOTENTIALPAYOUT_COEFFICIENT,
        DATEDIFF('second', base.REAL_START_TIME, base.CREATE_TIME) / 60.0 AS MINS_FROM_START
    FROM base
    WHERE base.INITIALPOTENTIALPAYOUT_COEFFICIENT IS NOT NULL
        AND base.CREATE_TIME IS NOT NULL
        AND base.INITIALPOTENTIALPAYOUT_COEFFICIENT > 1.0
        AND base.STAKE_EUR > 0
        AND DATEDIFF('second', base.REAL_START_TIME, base.CREATE_TIME) / 60.0 BETWEEN -60 AND 240
)
, band_base AS (
    SELECT bw.EVENT_ID, bw.EVENT_NAME, bw.REAL_START_TIME, bw.TOURNAMENT_NAME,
        bw.BUSINESS_DOMAIN, bw.SELECTIONS_MARKETLINE_LEGACY_NAME AS MARKET_NAME, bw.SPORT_NAME,
        TRIM(SPLIT_PART(bw.EVENT_NAME, '·', 1)) AS team1,
        TRIM(SPLIT_PART(bw.EVENT_NAME, '·', 2)) AS team2,
        bw.SELECTIONS_ODD_LEGACY_NAME AS outcome,
        bw.SELECTIONS_ODD_COEFFICIENT AS odds, bw.STAKE_EUR AS stake_eur,
        bw.STAKE_EUR * bw.SELECTIONS_ODD_COEFFICIENT AS ggr_stake_row,
        bw.TICKET_ID AS PLayer_ID,
        LEAST(FLOOR(bw.MINS_FROM_START / 10.0) * 10, 80) AS band_start_min,
        CONCAT(CAST(LEAST(FLOOR(bw.MINS_FROM_START / 10.0) * 10, 80) AS VARCHAR), ' to ',
               CAST(LEAST(FLOOR(bw.MINS_FROM_START / 10.0) * 10, 80) + 10 AS VARCHAR), ' mins') AS acceptance_time_band
    FROM base_windowed bw
)
, agg AS (
    SELECT EVENT_ID, EVENT_NAME, REAL_START_TIME, TOURNAMENT_NAME, BUSINESS_DOMAIN, MARKET_NAME, SPORT_NAME,
        team1, team2, outcome, acceptance_time_band, band_start_min,
        AVG(odds) AS odds, SUM(stake_eur) AS stake_eur, SUM(ggr_stake_row) AS ggr_stake_row,
        COUNT(DISTINCT PLAYER_ID) AS n_players, COUNT(DISTINCT PLAYER_ID) * AVG(odds) AS ggr_player_row
    FROM band_base
    GROUP BY EVENT_ID, EVENT_NAME, REAL_START_TIME, TOURNAMENT_NAME, BUSINESS_DOMAIN, MARKET_NAME, SPORT_NAME, team1, team2, outcome, acceptance_time_band, band_start_min
)
SELECT
    TO_CHAR(a.REAL_START_TIME + INTERVAL '1 HOUR', 'YYYY-MM-DD HH24:MI') AS REAL_START_TIME,
    a.EVENT_NAME, a.TOURNAMENT_NAME, a.BUSINESS_DOMAIN, a.MARKET_NAME, a.SPORT_NAME,
    a.team1 AS TEAM1, a.team2 AS TEAM2, a.outcome AS OUTCOME,
    a.acceptance_time_band AS ACCEPTANCE_TIME_BAND, a.odds,
    a.stake_eur AS STAKE_EUR, a.ggr_stake_row AS GGR_STAKE,
    a.n_players AS N_PLAYERS, a.ggr_player_row AS GGR_PLAYER,
    SUM(a.stake_eur) OVER (PARTITION BY a.EVENT_ID) AS STAKE_EUR_MATCH_TOTAL,
    SUM(a.n_players) OVER (PARTITION BY a.EVENT_ID) AS N_PLAYERS_MATCH_TOTAL
FROM agg a
WHERE a.SPORT_NAME IN ('Soccer', 'Tennis', 'Basketball')
ORDER BY REAL_START_TIME, a.band_start_min, STAKE_EUR DESC
);

In [ ]:
%%sql -r perf_comparison
-- Compare execution stats for both queries
SELECT
    QUERY_TAG,
    TOTAL_ELAPSED_TIME / 1000.0 AS elapsed_seconds,
    COMPILATION_TIME / 1000.0 AS compile_seconds,
    EXECUTION_TIME / 1000.0 AS exec_seconds,
    BYTES_SCANNED / 1024 / 1024 AS mb_scanned,
    BYTES_SPILLED_TO_LOCAL_STORAGE / 1024 / 1024 AS mb_spilled_local,
    BYTES_SPILLED_TO_REMOTE_STORAGE / 1024 / 1024 AS mb_spilled_remote,
    ROWS_PRODUCED,
    PARTITIONS_SCANNED,
    PARTITIONS_TOTAL
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY(
    END_TIME_RANGE_START => DATEADD(MINUTE, -15, CURRENT_TIMESTAMP()),
    RESULT_LIMIT => 20
))
WHERE QUERY_TAG IN ('perf_test_raw_optimised', 'perf_test_raw2_original')
  AND QUERY_TYPE = 'SELECT'
ORDER BY START_TIME DESC;